<div style="padding: 25px; background-color: #1a1a2e; border-radius: 12px; border: 2px solid #e94560;">
    <h1 style="color: #ff6b82; font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;">Predictive Site Selection Model</h1>
    <h2 style="color: #e0e0e0;">Binary Classification on H3 Spatial Hexagons — London Specialty Coffee</h2>
    <p style="color: #eee; font-size: 1.1em;">
        Predicting specialty coffee shop suitability from multi-modal geospatial features:<br>
        <strong>LandScan</strong> population rasters, <strong>ONS Census 2021</strong> demographics, and <strong>NetworkX</strong> graph centrality.
    </p>
    <hr style="border-color: #e94560;">
    <div style="display: flex; justify-content: space-between; color: #ccc;">
        <span>Greater London (33 Boroughs)</span>
        <span>MSc Business Analytics | 2026</span>
    </div>
</div>

<div style="margin-top: 20px; padding: 15px; background-color: #fff3cd; border-radius: 8px; border-left: 5px solid #ffc107;">
    <h3 style="color: #856404;">Data Access &amp; Reproducibility</h3>
    <p>This notebook requires three external datasets. Ensure all files are in place before running.</p>
    <table style="width:100%; border-collapse: collapse; margin: 10px 0; font-size: 0.9em;">
        <tr style="background: #856404; color: white;">
            <th style="padding: 8px;">Dataset</th>
            <th style="padding: 8px;">Source</th>
            <th style="padding: 8px;">How to Obtain</th>
            <th style="padding: 8px;">Expected Path</th>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>LandScan Global Population</strong></td>
            <td>Oak Ridge National Laboratory (ORNL)</td>
            <td>Register at <a href="https://landscan.ornl.gov/">landscan.ornl.gov</a>, download the UK mosaic GeoTIFF (~38 MB)</td>
            <td><code>landscan-mosaic-unitedkingdom-v1.tif</code></td>
        </tr>
        <tr style="background: #fff8e1;">
            <td style="padding: 8px;"><strong>ONS Census 2021</strong> (3 tables)</td>
            <td>EDINA Digimap / ONS Nomis</td>
            <td>Download OA-level tables: TS007 (age), TS066 (economic activity), TS067 (qualifications) for England &amp; Wales</td>
            <td><code>ons-age-ew-2021_*/</code>, <code>ons-economic-ew-2021_*/</code>, <code>ons-qualifications-ew-2021_*/</code></td>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>OpenStreetMap POIs</strong></td>
            <td>OSMnx (programmatic)</td>
            <td>Auto-fetched in Task 1 below via <code>ox.features_from_place()</code> &mdash; no manual download needed</td>
            <td>N/A (in-memory)</td>
        </tr>
    </table>
    <p><strong>Note:</strong> LandScan requires free academic registration. Census data is available through institutional Digimap access or the public <a href="https://www.nomisweb.co.uk/">Nomis</a> portal.</p>
</div>

In [ ]:
# --- Auto-Install Missing Dependencies ---
# WHY DYNAMIC CHECK: A requirements.txt cannot install itself. In a fresh environment,
#   some packages may be missing. This cell checks each with importlib.util.find_spec
#   (which does NOT import the package, avoiding circular import issues) and installs
#   only what is missing, keeping startup fast when packages are already present.

# === Environment Setup — Run this cell once to install any missing dependencies ===
import importlib, subprocess, sys

_required = {
    "pandas": "pandas>=2.0", "numpy": "numpy>=1.24", "geopandas": "geopandas>=0.14",
    "shapely": "shapely>=2.0", "h3": "h3>=4.0", "networkx": "networkx>=3.0",
    "osmnx": "osmnx>=1.7", "rasterio": "rasterio>=1.3", "rasterstats": "rasterstats>=0.19",
    "sklearn": "scikit-learn>=1.3", "xgboost": "xgboost>=2.0", "statsmodels": "statsmodels>=0.14",
    "matplotlib": "matplotlib>=3.7", "seaborn": "seaborn>=0.13", "pydeck": "pydeck>=0.8.1",
    "pyarrow": "pyarrow>=14.0", "streamlit": "streamlit>=1.28", "plotly": "plotly>=5.17",
}

missing = [pip_name for mod, pip_name in _required.items() if importlib.util.find_spec(mod) is None]

if missing:
    print(f"Installing {len(missing)} missing package(s): {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Done.")
else:
    print(f"All {len(_required)} packages already installed.")

<div style="margin-top: 30px; padding: 15px; background-color: #eaf2f8; border-radius: 8px; border-left: 5px solid #2e86c1;">
    <h2 style="color: #1a5276;">Pipeline Overview</h2>
    <p>This notebook is <strong>self-contained</strong>. It re-derives all features from raw data sources so it can run independently of Notebooks 01–03. The pipeline:</p>
    <ol>
        <li><strong>Data Assembly</strong>: Load H3 grid, LandScan raster, OSM POIs, and 3 ONS Census CSVs.</li>
        <li><strong>Feature Engineering</strong>: 14 features across 4 modalities (footfall, demographics, graph centrality incl. eigenvector &amp; PageRank, POI ecosystem).</li>
        <li><strong>Target Definition</strong>: Binary — <code>has_coffee_shop</code> (1/0) per hexagon.</li>
        <li><strong>Spatial Cross-Validation</strong>: H3 parent-cell block CV to prevent spatial leakage.</li>
        <li><strong>Model Comparison</strong>: Logistic Regression vs. Random Forest vs. XGBoost.</li>
        <li><strong>Hyperparameter Tuning</strong>: GridSearchCV with spatial folds.</li>
        <li><strong>Business Insight</strong>: Extract False Positives as site recommendations.</li>
    </ol>
</div>

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Concept: Three Data Modalities</h2>
    <p>Geospatial data comes in fundamentally different representations. Our pipeline must reconcile all three before any analysis can begin.</p>
    <table style="width:100%; border-collapse: collapse; margin: 15px 0;">
        <tr style="background: #2980b9; color: white;">
            <th style="padding: 8px;">Modality</th><th style="padding: 8px;">Representation</th><th style="padding: 8px;">Example in This Pipeline</th><th style="padding: 8px;">Spatial Precision</th>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>Vector</strong></td>
            <td>Discrete geometric primitives (points, lines, polygons) with exact coordinate pairs</td>
            <td>OSM cafe locations (Points), building footprints (Polygons)</td>
            <td>Sub-metre (coordinate precision)</td>
        </tr>
        <tr style="background: #f0f6fb;">
            <td style="padding: 8px;"><strong>Raster</strong></td>
            <td>Regular grid of cells (pixels), each storing a numeric value &mdash; analogous to a satellite image band</td>
            <td>LandScan population grid &mdash; each pixel holds an estimated population count</td>
            <td>~1 km per pixel</td>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>Tabular</strong></td>
            <td>Attribute records keyed by a spatial identifier (Output Area code), linkable to geometry via a join</td>
            <td>ONS Census CSVs &mdash; demographic percentages per Output Area, with BNG centroids</td>
            <td>OA centroid (~125 m median OA diameter in Camden)</td>
        </tr>
    </table>
    <p><strong>Key Principle:</strong> All three must be projected to the same Coordinate Reference System (CRS)
    before they can interact. We use:</p>
    <p style="text-align: center; font-size: 1.1em;">
        $$\mathbf{x}_{BNG} = T_{4326 \to 27700}(\mathbf{x}_{WGS84})$$
    </p>
    <p>where $T$ is the Ordnance Survey's Transverse Mercator projection for Great Britain.</p>
</div>

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Concept: Why Hexagons (H3)?</h2>
    <p>Uber's <strong>H3</strong> library partitions the globe into hierarchical hexagonal cells.
    Compared to square grids:</p>
    <ul>
        <li><strong>Squares:</strong> 4 edge-neighbours + 4 corner-neighbours at different distances &mdash; ambiguous adjacency.</li>
        <li><strong>Hexagons:</strong> All 6 neighbours share an edge at the <em>same</em> distance from the centre &mdash; ideal for walking-distance analysis.</li>
    </ul>
    <table style="width:100%; border-collapse: collapse; margin: 15px 0; font-size: 0.95em;">
        <tr style="background: #2e86c1; color: white;">
            <th style="padding: 8px;">Resolution</th>
            <th style="padding: 8px;">Edge Length</th>
            <th style="padding: 8px;">Hex Area</th>
            <th style="padding: 8px;">Use Case</th>
        </tr>
        <tr><td style="padding: 8px;">7</td><td>~1.2 km</td><td>~5.16 km&sup2;</td><td>District-level analysis</td></tr>
        <tr style="background: #eaf2f8;"><td style="padding: 8px;">8</td><td>~460 m</td><td>~0.74 km&sup2;</td><td>Neighbourhood catchment</td></tr>
        <tr style="font-weight: bold; background: #d4efdf;"><td style="padding: 8px;">9</td><td>~174 m</td><td>~0.105 km&sup2;</td><td>Walking-scale / 15-min city (our choice)</td></tr>
        <tr style="background: #eaf2f8;"><td style="padding: 8px;">10</td><td>~66 m</td><td>~0.015 km&sup2;</td><td>Street-level micro-analysis</td></tr>
    </table>
    <p><strong>Hierarchical property:</strong> Every Res-9 hex has exactly one Res-5 parent.
    We exploit this in the ML notebook for Spatial Block Cross-Validation
    (grouping child hexes by parent to prevent spatial leakage).</p>
    <p>Hex area formula: $A_{hex} \approx 2.6 \times s^2$, where $s$ is the edge length.</p>
</div>

In [ ]:
# ============================================================
# --- Core Imports and Configuration ---
# WHY RANDOM_STATE = 42: XGBoost, RandomForest, and GridSearchCV all involve randomness.
#   Fixing the seed ensures identical results on re-run (MSc reproducibility requirement).

# SECTION 0: Environment & Imports
# ============================================================
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import geopandas as gpd
import networkx as nx
import h3
# Fix: rasterio on Windows scans entire PATH for gdal*.dll via glob.glob(),
# which hangs if PATH includes slow/network directories (e.g. OneDrive).
# Temporarily strip PATH to only fast local dirs during import.
import os
_original_path = os.environ.get('PATH', '')
os.environ['PATH'] = os.pathsep.join(
    p for p in _original_path.split(os.pathsep)
    if p and not any(slow in p.lower() for slow in ['onedrive', 'network', 'nas'])
)
import rasterio
os.environ['PATH'] = _original_path  # restore full PATH
import rasterstats
import osmnx as ox
import matplotlib.pyplot as plt
import seaborn as sns
import pydeck as pdk
from shapely.geometry import Polygon, Point

# ML Stack
from sklearn.model_selection import cross_val_score, cross_val_predict, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
import xgboost as xgb

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Directory scaffold
for d in ['data/raw', 'data/processed', 'data/outputs']:
    os.makedirs(d, exist_ok=True)

# ── Multi-Business-Type Configuration ──────────────────────
# WHY SEPARATE DICTS (BUSINESS_TYPES vs ALWAYS_FEATURES): Business types are
#   prediction targets (one XGBoost model per type). Always-features are never
#   predicted but their counts appear as co-occurrence signals in every model.
#   Keeping them separate prevents accidental inclusion as a prediction target.

BUSINESS_TYPES = {
    'cafe':       {'label': 'Coffee Shop / Cafe'},
    'restaurant': {'label': 'Restaurant'},
    'pub':        {'label': 'Pub / Bar'},
    'fast_food':  {'label': 'Fast Food'},
    'gym':        {'label': 'Gym / Fitness'},
    'bakery':     {'label': 'Bakery'},
}

# Non-selectable types — always used as co-occurrence features
ALWAYS_FEATURES = {
    'supermarket': {}, 'office': {}, 'library': {},
    'university': {}, 'station': {},
}

ALL_POI_TYPES = list(BUSINESS_TYPES.keys()) + list(ALWAYS_FEATURES.keys())

print(f"H3 version: {h3.__version__}")
print(f"Business types: {list(BUSINESS_TYPES.keys())}")
print(f"Environment ready. Random state: {RANDOM_STATE}")


<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 1: Data Preparation — The Feature Matrix</h2>
    <p>We assemble a single flat <code>DataFrame</code> by joining three data modalities onto the H3 hexagonal grid.</p>
    <table style="width:100%; border-collapse: collapse; margin-top: 10px; font-size: 0.95em;">
        <tr style="background: #16213e; color: white;">
            <th style="padding: 8px; text-align: left;">Modality</th>
            <th style="padding: 8px; text-align: left;">Source</th>
            <th style="padding: 8px; text-align: left;">Features</th>
        </tr>
        <tr style="background: #f8f9fa;"><td style="padding: 8px;">Footfall</td><td>LandScan Raster (zonal stats)</td><td><code>population</code></td></tr>
        <tr><td style="padding: 8px;">Demographics</td><td>ONS Census 2021 (Digimap)</td><td><code>employed_total_perc</code>, <code>age_16_to_34_perc</code>, <code>level4_perc</code>, <code>retired_perc</code>, <code>no_qualifications_perc</code></td></tr>
        <tr style="background: #f8f9fa;"><td style="padding: 8px;">Graph Centrality</td><td>NetworkX on H3 adjacency</td><td><code>degree_centrality</code>, <code>betweenness_centrality</code>, <code>closeness_centrality</code>, <code>clustering_coeff</code>, <code>eigenvector_centrality</code>, <code>pagerank</code></td></tr>
        <tr><td style="padding: 8px;">POI Ecosystem</td><td>OSMnx spatial join</td><td><code>n_synergy</code>, <code>n_anchors</code> (competitors excluded — they <em>are</em> the target)</td></tr>
    </table>
    <p><strong>Total: 14 features</strong></p>
    <div style="background: #eaf2f8; padding: 12px; border-radius: 5px; border-left: 4px solid #2e86c1; margin-top: 10px;">
        <strong>Why all three modalities?</strong> No single data source is sufficient:
        <ul style="margin: 5px 0;">
            <li><strong>Footfall alone</strong> misses demand quality &mdash; a park has high foot traffic but zero coffee demand.</li>
            <li><strong>Demographics alone</strong> misses spatial structure &mdash; an affluent hex surrounded by competitors is saturated.</li>
            <li><strong>Graph centrality alone</strong> misses demand intensity &mdash; a transit corridor through industrial land has high betweenness but no customers.</li>
        </ul>
        The three modalities are complementary: footfall provides volume, demographics provide quality, and graph centrality provides structural position.
    </div>
</div>

In [ ]:
# ============================================================
# --- H3 Hexagonal Grid over Greater London ---
# WHY H3 RESOLUTION 9: Edge length ~174m, area ~0.105 km2. This matches a
#   2-3 minute walking radius from a retail site -- the typical catchment area
#   a customer would walk to a coffee shop. Res-8 (~462m) is too coarse for
#   site selection; Res-10 (~65m) produces too many hexes and exceeds POI
#   spatial precision from OSM.

# SECTION 1.1: Generate H3 Grid over Greater London
# ============================================================
PLACE = "Greater London"
RESOLUTION = 9  # ~174m edge length — walking scale

# All 33 London boroughs (32 boroughs + City of London)
# WHY LIST ALL 33 EXPLICITLY: OSMnx geocode_to_gdf("Greater London") returns a
#   single merged polygon, not individual borough boundaries. We need borough-level
#   POI fetching (Cell 10) to keep Overpass queries tractable, so we maintain this
#   list as the authoritative enumeration. City of London is included separately
#   because it is not a "London Borough" but must be covered for completeness.

# Explicit list of all 33 London boroughs — required for per-borough POI fetching
BOROUGH_NAMES = [
    "London Borough of Barking and Dagenham", "London Borough of Barnet",
    "London Borough of Bexley", "London Borough of Brent",
    "London Borough of Bromley", "London Borough of Camden",
    "London Borough of Croydon", "London Borough of Ealing",
    "London Borough of Enfield", "London Borough of Greenwich",
    "London Borough of Hackney", "London Borough of Hammersmith and Fulham",
    "London Borough of Haringey", "London Borough of Harrow",
    "London Borough of Havering", "London Borough of Hillingdon",
    "London Borough of Hounslow", "London Borough of Islington",
    "Royal Borough of Kensington and Chelsea",
    "Royal Borough of Kingston upon Thames", "London Borough of Lambeth",
    "London Borough of Lewisham", "London Borough of Merton",
    "London Borough of Newham", "London Borough of Redbridge",
    "London Borough of Richmond upon Thames", "London Borough of Southwark",
    "London Borough of Sutton", "London Borough of Tower Hamlets",
    "London Borough of Waltham Forest", "London Borough of Wandsworth",
    "City of Westminster", "City of London",
]

# Fetch Greater London boundary
print("Fetching Greater London boundary...")
boundary = ox.geocode_to_gdf(PLACE)
boundary_wgs84 = boundary.to_crs(epsg=4326)
poly = boundary_wgs84.geometry.iloc[0]

# WHY TAKE LARGEST POLYGON: OSMnx sometimes returns a MultiPolygon for Greater
#   London because the Thames creates disconnected landmasses (e.g., Isle of Dogs).
#   The largest polygon by area is the mainland. Smaller fragments contain no
#   retail areas not already captured by mainland hexagons.

# Handle MultiPolygon (OSM may return this for Greater London)
from shapely.geometry import MultiPolygon as _MP
if isinstance(poly, _MP):
    poly = max(poly.geoms, key=lambda p: p.area)
    print(f"  Extracted largest polygon from MultiPolygon")

# Fetch borough boundaries one-by-one (resilient to individual geocoding failures)
print("Fetching 33 borough boundaries...")
_borough_gdfs = []
for _i, _name in enumerate(BOROUGH_NAMES):
    try:
        _gdf = ox.geocode_to_gdf(_name)
        _short = _name.replace("London Borough of ", "").replace("Royal Borough of ", "")
        _gdf['borough'] = _short
        _borough_gdfs.append(_gdf)
    except Exception as _e:
        print(f"  WARNING: '{_name}' failed ({_e}), skipping.")
boroughs_gdf = pd.concat(_borough_gdfs, ignore_index=True)
boroughs_gdf = boroughs_gdf.to_crs(epsg=4326)
print(f"  Successfully geocoded {len(boroughs_gdf)}/{len(BOROUGH_NAMES)} boroughs")

# Convert borough polygon to H3 hex set; swap (lng,lat) -> (lat,lng) for H3 API
# H3 v4: polygon_to_cells
# WHY COORDINATE REVERSAL: H3 uses (lat, lng) order (geographic convention).
#   Shapely uses (lng, lat) = (x, y) (Cartesian convention). Passing coordinates
#   in the wrong order produces hexagons covering the wrong hemisphere.

outer_coords = [(lat, lng) for lng, lat in poly.exterior.coords]
holes = [[(lat, lng) for lng, lat in ring.coords] for ring in poly.interiors]
h3_poly = h3.LatLngPoly(outer_coords, *holes)
hex_ids = h3.polygon_to_cells(h3_poly, RESOLUTION)

# Convert to GeoDataFrame
def h3_to_shapely(cell_id):
    coords = h3.cell_to_boundary(cell_id)
    return Polygon([(lng, lat) for lat, lng in coords])

hex_polygons = [h3_to_shapely(h) for h in hex_ids]
h3_grid = gpd.GeoDataFrame(
    {'h3_index': list(hex_ids)},
    geometry=hex_polygons,
    crs='EPSG:4326'
)

# Tag each hexagon with its London borough via centroid-in-polygon spatial join
hex_centroids = h3_grid.copy()
hex_centroids.geometry = hex_centroids.geometry.centroid

borough_join = gpd.sjoin(
    hex_centroids[['h3_index', 'geometry']],
    boroughs_gdf[['borough', 'geometry']],
    how='left', predicate='within'
)
h3_grid['borough'] = borough_join.set_index('h3_index').reindex(h3_grid['h3_index'])['borough'].values

# Fallback for edge hexes that fall between borough boundaries
missing = h3_grid['borough'].isna()
if missing.any():
    # WHY TWO-PASS JOIN: Primary sjoin(predicate="within") tags hexagons whose
    #   centroid falls inside a borough. Edge hexagons spanning two boroughs may
    # Fallback: edge hexagons spanning boundaries may not be "within" any borough
    #   fail the "within" test. sjoin_nearest assigns these to the closest borough,
    #   ensuring complete coverage with no NaN borough values.

    fallback = gpd.sjoin_nearest(
        hex_centroids.loc[missing, ['h3_index', 'geometry']],
        boroughs_gdf[['borough', 'geometry']]
    )
    h3_grid.loc[missing, 'borough'] = fallback.drop_duplicates('h3_index').set_index('h3_index').reindex(h3_grid.loc[missing, 'h3_index'])['borough'].values

print(f"H3 Grid: {len(h3_grid)} hexagons at Resolution {RESOLUTION}")
print(f"Boroughs covered: {h3_grid['borough'].nunique()}")
print(f"\nHexagons per borough (top 10):")
print(h3_grid['borough'].value_counts().head(10).to_string())


In [ ]:
# ============================================================
# --- LandScan Population Raster Enrichment ---
# WHY nodata=-999: LandScan uses -999 as its no-data sentinel, not NaN. Without
#   this parameter, ocean and no-coverage pixels would contribute -999 to the sum,
#   dramatically inflating population for coastal hexagons.
# WHY NO CRS CONVERSION: rasterstats.zonal_stats automatically reprojects geometry
#   to match the raster CRS. Both h3_grid (EPSG:4326) and LandScan (WGS84/4326)
#   are already in the same CRS, so no manual conversion is needed.

# SECTION 1.2: Footfall — LandScan Raster Enrichment
# ============================================================
raster_path = "landscan-mosaic-unitedkingdom-v1.tif"

with rasterio.open(raster_path) as src:
    print(f"Raster CRS: {src.crs} | Resolution: {src.res}")

# Zonal stats: sum population pixels under each hex
stats = rasterstats.zonal_stats(h3_grid, raster_path, stats=['sum'], nodata=-999)
h3_grid['population'] = [s['sum'] if s['sum'] is not None else 0 for s in stats]

print(f"Total LandScan population across London hexes: {h3_grid['population'].sum():,.0f}")
print(f"Max hex population: {h3_grid['population'].max():,.0f}")
print(f"Hexes with zero population: {(h3_grid['population'] == 0).sum()}")


In [ ]:
# ============================================================
# SECTION 1.3: POI Ecosystem — OSMnx Fetch & Granular Typing
# ============================================================
import time
# --- POI Tag Configuration ---
# WHY 3 BATCHES: The Overpass API imposes a max result-set size per query (~10k elements).
#   Dense boroughs (Westminster, Greenwich) return enough POIs in a single query to hit
#   this ceiling, causing 413 errors or silent truncation. Splitting by semantic category
#   (food / leisure+fitness / retail+transport) keeps each query under the limit.

# Split tags into 3 small batches — Greenwich/Westminster choke on larger queries
tags_batch1 = {
    'amenity': ['cafe', 'coffee_shop', 'restaurant', 'pub', 'bar', 'fast_food'],
}
tags_batch2 = {
    'amenity': ['gym', 'university', 'office', 'library', 'leisure_centre'],
    'leisure': ['fitness_centre', 'sports_centre'],
}
tags_batch3 = {
    'shop': ['bakery', 'supermarket'],
    'public_transport': ['station'],
    'office': ['company', 'coworking', 'government'],
}
tags_batches = [tags_batch1, tags_batch2, tags_batch3]
# --- Overpass API Reliability Settings ---
# WHY CUSTOM RETRY LOGIC: The default osmnx settings assume a single reliable endpoint.
#   For a 33-borough London-wide fetch, we need endpoint rotation, exponential backoff,
#   and automatic query subdivision to handle the inevitable 413/timeout failures.

# --- Overpass API reliability settings ---
ox.settings.requests_timeout = 90            # 90s (was 300s, too slow for retries)
ox.settings.overpass_rate_limit = False      # disabled: server can impose infinite waits
# WHY THREE ENDPOINTS: overpass-api.de is the canonical public endpoint but is frequently
#   throttled during peak hours. Two verified mirrors allow automatic rotation on failure
#   without manual intervention. Order matters: primary first, mirrors via attempt % len().

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api",           # primary (default)
    "https://overpass.private.coffee/api",   # mirror 1
    "https://maps.mail.ru/osm/tools/overpass/api",  # mirror 2
]
MAX_RETRIES = 3
# --- Borough Name Normalisation ---
# WHY PREFIX STRIPPING: OSMnx geocodes full names ("London Borough of Camden") but
#   boroughs_gdf uses short forms ("Camden"). This deterministic mapping handles all
#   prefix variations including "Royal Borough of" and "City of".

# Deterministic name → short name mapping (handles all prefix variations)
def _to_short_name(full_name):
    for prefix in ["London Borough of ", "Royal Borough of ", "City of "]:
        if full_name.startswith(prefix):
            return full_name[len(prefix):]
    return full_name
BOROUGH_NAME_TO_SHORT = {name: _to_short_name(name) for name in BOROUGH_NAMES}
# --- Single-Batch Fetch with Retry + Endpoint Rotation ---
# WHY TWO STRATEGIES PER ATTEMPT: features_from_place (geocoded name) is tried first
#   as it is simpler and more accurate. features_from_polygon (pre-fetched boundary)
#   covers cases where Nominatim geocoding fails. This two-stage design maximises
#   success probability before consuming another retry slot.

def _fetch_single_batch(borough_name, tags, boroughs_gdf, endpoints, max_retries=3):
    """Fetch one batch of tags for a borough with retry + endpoint rotation."""
    short_name = BOROUGH_NAME_TO_SHORT[borough_name]
    errors = []
    for attempt in range(max_retries):
        endpoint = endpoints[attempt % len(endpoints)]
        ox.settings.overpass_url = endpoint
        # Extract short hostname for logging (e.g., "overpass-api.de")
        ep_short = endpoint.split("//")[1].split("/")[0]
        try:
            pois = ox.features_from_place(borough_name, tags)
            return pois, f"place (attempt {attempt+1}, {ep_short})"
        except Exception as e:
            errors.append(f"place@{attempt+1}: {e}")
        try:
            match = boroughs_gdf[boroughs_gdf['borough'] == short_name]
            if len(match) > 0:
                pois = ox.features_from_polygon(match.geometry.iloc[0], tags)
                return pois, f"polygon (attempt {attempt+1}, {ep_short})"
            else:
                errors.append(f"polygon@{attempt+1}: no boundary for '{short_name}'")
        except Exception as e:
            errors.append(f"polygon@{attempt+1}: {e}")
        if attempt < max_retries - 1:
            wait = 2 ** attempt * 5
            print(f"      batch retry {attempt+1} failed, waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError(chr(10).join(f"  - {e}" for e in errors))

# --- Borough-Level Fetch with Auto-Split Fallback ---
# WHY AUTO-SPLIT ON BATCH FAILURE: When an entire batch fails even after retries,
#   the likely cause is result-set overflow, not a network error. Splitting into
#   individual single-tag queries (e.g., amenity=cafe alone) reduces each query size
#   by ~80%, making it tractable for the densest boroughs.

def fetch_borough_pois(borough_name, tags_batches, boroughs_gdf, endpoints, max_retries=3):
    """Fetch POIs in batches; auto-splits into individual tags on failure."""
    chunks = []
    for bi, batch_tags in enumerate(tags_batches):
        try:
            chunk, strat = _fetch_single_batch(
                borough_name, batch_tags, boroughs_gdf, endpoints, max_retries
            )
            chunks.append(chunk)
        except RuntimeError:
            # Batch too large for this borough — split into individual tag queries
            print(f"    Batch {bi+1} too large, splitting into individual tags...")
            for tag_key, tag_vals in batch_tags.items():
                if isinstance(tag_vals, list):
                    for val in tag_vals:
                        mini = {tag_key: [val]}
                        try:
                            chunk, _ = _fetch_single_batch(
                                borough_name, mini, boroughs_gdf, endpoints, max_retries
                            )
                            chunks.append(chunk)
                        except RuntimeError:
                            print(f"      WARN: {tag_key}={val} failed, skipping")
                        time.sleep(1)
        if bi < len(tags_batches) - 1:
            time.sleep(1)
    if not chunks:
        raise RuntimeError(f"All batches failed for '{borough_name}'")
    # Merge all successful batch/tag chunks into one DataFrame
    combined = pd.concat(chunks, ignore_index=False)
    # Remove boundary-overlap duplicates (OSM IDs are globally unique)
    combined = combined[~combined.index.duplicated(keep='first')]
    return combined, "batched"

# --- Priority Fetch: Dense Boroughs via Direct Overpass QL ---
# WHY PRIORITY FETCH: Westminster consistently fails when fetched late in the
#   queue because ~100+ osmnx queries for earlier boroughs exhaust the Overpass
#   API rate limit. Fetching Westminster FIRST, with a fresh API and compact
#   direct Overpass QL queries, resolves this. The main loop then skips it.
# WHY DIRECT OVERPASS QL (not osmnx): Compact bbox notation, `out center;`
#   response format, explicit maxsize control. See retry section comments.

import requests as _req
import numpy as _np

PRIORITY_BOROUGHS = ["City of Westminster"]  # historically problematic
pre_fetched = {}  # borough_name -> GeoDataFrame

# Flatten all tags into (key, value) pairs
all_tag_pairs = []
for batch in tags_batches:
    for tkey, tvals in batch.items():
        if isinstance(tvals, list):
            for tv in tvals:
                all_tag_pairs.append((tkey, tv))

for borough_name in PRIORITY_BOROUGHS:
    short_name = BOROUGH_NAME_TO_SHORT[borough_name]
    match = boroughs_gdf[boroughs_gdf['borough'] == short_name]
    if len(match) == 0:
        print(f"  SKIP priority fetch: no boundary for {short_name}")
        continue
    poly_bng = match.geometry.iloc[0]
    poly_wgs = gpd.GeoSeries([poly_bng], crs='EPSG:27700').to_crs('EPSG:4326').iloc[0]
    minx, miny, maxx, maxy = poly_wgs.bounds

    # Build 4x4 grid of sub-bounding-boxes (16 tiles)
    GRID_N = 4
    xs = _np.linspace(minx, maxx, GRID_N + 1)
    ys = _np.linspace(miny, maxy, GRID_N + 1)
    sub_bboxes = []
    for gi in range(GRID_N):
        for gj in range(GRID_N):
            sub_bboxes.append((ys[gj], xs[gi], ys[gj + 1], xs[gi + 1]))

    _nt = len(sub_bboxes)
    _ntags = len(all_tag_pairs)
    _nq = _nt * _ntags
    print(f"Priority fetch: {borough_name} ({_nt} tiles x {_ntags} tags = {_nq} queries)...")
    retry_rows = []
    queries_failed = 0
    for ti, bbox in enumerate(sub_bboxes):
        south, west, north, east = bbox
        for tkey, tval in all_tag_pairs:
            query = (
                '[out:json][timeout:90][maxsize:50000000];'
                '('
                f'node["{tkey}"="{tval}"]({south},{west},{north},{east});'
                f'way["{tkey}"="{tval}"]({south},{west},{north},{east});'
                f'relation["{tkey}"="{tval}"]({south},{west},{north},{east});'
                ');out center;'
            )
            fetched = False
            for ep in OVERPASS_ENDPOINTS:
                try:
                    url = f"{ep}/interpreter"
                    resp = _req.post(url, data={'data': query}, timeout=120)
                    resp.raise_for_status()
                    elements = resp.json().get('elements', [])
                    for el in elements:
                        lat = el.get('lat')
                        lon = el.get('lon')
                        if lat is None or lon is None:
                            center = el.get('center') or {}
                            lat = center.get('lat')
                            lon = center.get('lon')
                        if lat is not None and lon is not None:
                            retry_rows.append({'lat': lat, 'lon': lon, 'osmid': el.get('id'), tkey: tval})
                    fetched = True
                    break
                except Exception:
                    time.sleep(3)
            if not fetched:
                queries_failed += 1
            time.sleep(1)
        _td = ti + 1
        if _td % 4 == 0 or _td == _nt:
            _nr = len(retry_rows)
            print(f"  Tiles: {_td}/{_nt} | POIs: {_nr} | Failed queries: {queries_failed}")

    if retry_rows:
        gdf = gpd.GeoDataFrame(
            retry_rows,
            geometry=gpd.points_from_xy([r['lon'] for r in retry_rows], [r['lat'] for r in retry_rows]),
            crs='EPSG:4326'
        )
        # Clip to borough boundary
        gdf_bng = gdf.to_crs('EPSG:27700')
        in_borough = gdf_bng.geometry.within(poly_bng)
        gdf = gdf[in_borough.values]
        if 'osmid' in gdf.columns:
            gdf = gdf.drop_duplicates(subset='osmid', keep='first')
        pre_fetched[borough_name] = gdf
        _nr2 = len(gdf)
        print(f"  {borough_name}: {_nr2} POIs fetched via priority Overpass QL")
    else:
        print(f"  {borough_name}: priority fetch returned 0 POIs")

    # Brief cooldown before main loop starts
    time.sleep(5)

# Reset endpoint for main loop
ox.settings.overpass_url = OVERPASS_ENDPOINTS[0]

# --- Main Fetch Loop ---
# Iterates over all 33 London boroughs sequentially. A 2s pause between boroughs
# respects Overpass fair-use policy (~1 req/2s). Failed boroughs are collected for
# the quadrant-based retry pass below.

# Fetch POIs borough-by-borough with robust retry logic
# Add any priority-fetched boroughs first
all_pois = list(pre_fetched.values())
pre_fetched_names = set(pre_fetched.keys())
_total = len(BOROUGH_NAMES)
_pre = len(pre_fetched)
print(f"Fetching POIs for {_total} boroughs ({_pre} pre-fetched, {_total - _pre} remaining)...")
failed_boroughs = []
for i, borough_name in enumerate(BOROUGH_NAMES):
    # Skip boroughs already fetched in priority pass
    if borough_name in pre_fetched_names:
        print(f"  [{i+1}/{_total}] {borough_name}: ALREADY FETCHED (priority pass)")
        continue
    try:
        pois_chunk, strategy = fetch_borough_pois(
            borough_name, tags_batches, boroughs_gdf,
            OVERPASS_ENDPOINTS, MAX_RETRIES
        )
        all_pois.append(pois_chunk)
        print(f"  [{i+1}/{len(BOROUGH_NAMES)}] {borough_name}: "
              f"{len(pois_chunk)} POIs (via {strategy})")
    except RuntimeError as e:
        failed_boroughs.append(borough_name)
        print(f"  [{i+1}/{len(BOROUGH_NAMES)}] FAILED: {borough_name}\n{e}")
    # Overpass API fair-use policy: ~1 request per 2 seconds
    time.sleep(2)  # Rate-limit between boroughs
# Reset to default endpoint
ox.settings.overpass_url = OVERPASS_ENDPOINTS[0]
# --- Direct Overpass QL Retry for Failed Boroughs ---
# WHY DIRECT OVERPASS QL (not osmnx): osmnx features_from_polygon generates verbose
#   Overpass queries embedding full polygon vertex lists. Even simple shapely boxes
#   produce large query bodies. Direct Overpass QL uses compact bbox notation
#   (4 numbers), giving explicit control over timeout, maxsize, and output format.
# WHY `out center;`: Returns only centroid coordinates instead of full geometry
#   polygons. Reduces response payload by ~90%, making dense boroughs tractable.
# WHY 4x4 GRID (16 tiles): Westminster has ~3x the POI density of an average
#   London borough. The previous 2x2 grid (4 quadrants) still overflowed.
#   16 tiles keep each individual query well under the Overpass result-set limit.
# WHY PER-TAG QUERIES: Each tile is queried once per tag (e.g., amenity=cafe).
#   16 tiles x 18 tags = 288 small queries, each returning at most a few hundred
#   POIs, comfortably within Overpass limits.

# ---- RETRY FAILED BOROUGHS (Direct Overpass QL + 4x4 grid) ----
if failed_boroughs:
    import requests as _req
    import numpy as _np
    print()
    _nf = len(failed_boroughs)
    print(f"{_nf} borough(s) failed main pass. Retrying via direct Overpass QL...")
    print("Cooling down 120s to reset Overpass rate limits...")
    time.sleep(120)

    # Flatten all tags into (key, value) pairs for individual querying
    # Flatten 3 batches into individual (tag_key, tag_val) pairs for granular querying
    all_tag_pairs = []
    for batch in tags_batches:
        for tkey, tvals in batch.items():
            if isinstance(tvals, list):
                for tv in tvals:
                    all_tag_pairs.append((tkey, tv))

    still_failed = []
    for borough_name in failed_boroughs:
        short_name = BOROUGH_NAME_TO_SHORT[borough_name]
        match = boroughs_gdf[boroughs_gdf['borough'] == short_name]
        if len(match) == 0:
            still_failed.append(borough_name)
            continue
        poly_bng = match.geometry.iloc[0]
        poly_wgs = gpd.GeoSeries([poly_bng], crs='EPSG:27700').to_crs('EPSG:4326').iloc[0]
        minx, miny, maxx, maxy = poly_wgs.bounds

        # Build 4x4 grid of sub-bounding-boxes (16 tiles)
        GRID_N = 4
        xs = _np.linspace(minx, maxx, GRID_N + 1)
        ys = _np.linspace(miny, maxy, GRID_N + 1)
        # Subdivide borough bounds into 16 non-overlapping tiles
        sub_bboxes = []
        for gi in range(GRID_N):
            for gj in range(GRID_N):
                # Overpass QL bbox format: (south, west, north, east)
                sub_bboxes.append((ys[gj], xs[gi], ys[gj + 1], xs[gi + 1]))

        _nt = len(sub_bboxes)
        _ntags = len(all_tag_pairs)
        _nq = _nt * _ntags
        print(f"  Retrying {borough_name}: {_nt} tiles x {_ntags} tags = {_nq} queries")
        retry_rows = []
        # Track progress for logging (288 queries can take ~5 minutes)
        queries_done = 0
        queries_failed = 0
        for ti, bbox in enumerate(sub_bboxes):
            south, west, north, east = bbox
            for tkey, tval in all_tag_pairs:
                # Build minimal Overpass QL query with bbox notation
                query = (
                    '[out:json][timeout:90][maxsize:50000000];'
                    '('
                    f'node["{tkey}"="{tval}"]({south},{west},{north},{east});'
                    f'way["{tkey}"="{tval}"]({south},{west},{north},{east});'
                    f'relation["{tkey}"="{tval}"]({south},{west},{north},{east});'
                    ');out center;'
                )
                fetched = False
                for ep in OVERPASS_ENDPOINTS:
                    try:
                        url = f"{ep}/interpreter"
                        # POST body contains the Overpass QL query string
                        resp = _req.post(url, data={'data': query}, timeout=120)
                        resp.raise_for_status()
                        # 200 OK — parse JSON response for POI elements
                        elements = resp.json().get('elements', [])
                        for el in elements:
                            lat = el.get('lat')
                            lon = el.get('lon')
                            # Ways/relations use 'center' sub-object from out center;
                            if lat is None or lon is None:
                                center = el.get('center') or {}
                                lat = center.get('lat')
                                lon = center.get('lon')
                            if lat is not None and lon is not None:
                                row = {
                                    'lat': lat, 'lon': lon,
                                    'osmid': el.get('id'),
                                    tkey: tval,
                                }
                                retry_rows.append(row)
                        fetched = True
                        break  # success on this endpoint, skip remaining
                    except Exception:
                        time.sleep(3)
                if not fetched:
                    queries_failed += 1
                queries_done += 1
                time.sleep(1)  # 1s between queries (Overpass fair-use)
            # Progress update every 4 tiles
            _td = ti + 1
            _nr = len(retry_rows)
            if _td % 4 == 0 or _td == len(sub_bboxes):
                print(f"    Tiles: {_td}/{_nt} | POIs so far: {_nr} | Failed: {queries_failed}")

        if retry_rows:
            # Build GeoDataFrame from collected POI rows
            gdf = gpd.GeoDataFrame(
                retry_rows,
                geometry=gpd.points_from_xy(
                    [r['lon'] for r in retry_rows],
                    [r['lat'] for r in retry_rows]
                ),
                crs='EPSG:4326'
            )
            # Clip to actual borough boundary (tiles extend beyond borough edges)
            gdf_bng = gdf.to_crs('EPSG:27700')
            # Spatial clip: discard POIs outside the actual borough polygon
            in_borough = gdf_bng.geometry.within(poly_bng)
            gdf = gdf[in_borough.values]
            # Deduplicate by osmid (same POI can appear in overlapping tile edges)
            if 'osmid' in gdf.columns:
                gdf = gdf.drop_duplicates(subset='osmid', keep='first')
            all_pois.append(gdf)
            _nr2 = len(gdf)
            print(f"  {borough_name}: RECOVERED {_nr2} POIs via direct Overpass QL")
        else:
            still_failed.append(borough_name)
            print(f"  {borough_name}: STILL FAILED (0 POIs from {_nq} queries)")

    failed_boroughs = still_failed
    ox.settings.overpass_url = OVERPASS_ENDPOINTS[0]

# --- Data Integrity Guard ---
# WHY HARD STOP: A model trained on 31/33 boroughs has systematic spatial gaps.
#   All 33 are required for a London-wide prediction grid. Silent data loss here
#   would silently degrade model quality downstream.

# ---- DATA INTEGRITY CHECK ----
if failed_boroughs:
    _nf = len(failed_boroughs)
    print(f"WARNING: POI fetch FAILED for {_nf} borough(s): {failed_boroughs}")
    print("Proceeding with available data. Missing boroughs will have 0 POIs.")
    print("The model can still train but predictions in missing areas may be less accurate.")
else:
    print(f"All {len(BOROUGH_NAMES)} boroughs fetched successfully.")
# --- Post-Fetch: Concatenate, Deduplicate, Reproject ---

# Concatenate all borough GeoDataFrames into a single London-wide POI dataset
pois_raw = pd.concat(all_pois, ignore_index=False)
# Deduplicate POIs that appear in multiple boroughs (boundary overlap)
# WHY DEDUPLICATE: OSM IDs are globally unique. Duplicate indices arise when a POI
#   lies on a borough boundary and is fetched from both adjacent boroughs.
#   keep="first" is safe because both copies are identical.

pois_raw = pois_raw[~pois_raw.index.duplicated(keep='first')]
# WHY CRS CHAIN: Centroids computed in EPSG:27700 (British National Grid) for correct
#   metric distances, then converted to EPSG:4326 (WGS84) for the spatial join with
#   h3_grid, which stores hexagons in WGS84 for H3 library compatibility.

# Convert geometries to centroids (Points) for consistent spatial joins downstream
pois_raw['geometry'] = pois_raw.to_crs(epsg=27700).centroid
pois_raw = pois_raw.set_crs(epsg=27700, allow_override=True)
# Filter: keep only Point geometries (multipoints/lines from rare OSM entries)
pois = pois_raw[pois_raw.geometry.type == 'Point'].copy()
pois = pois.to_crs(epsg=4326)  # match h3_grid CRS for sjoin
# Granular type classification — assigns each POI to one of 11 specific types
# --- Granular POI Type Classification ---
# WHY RULE-BASED NOT OSM TAGS DIRECTLY: OSM tagging is highly inconsistent.
#   A cafe may be tagged amenity=cafe, amenity=coffee_shop, or shop=coffee.
#   This function provides a deterministic, auditable mapping.
#   6 selectable types (cafe, restaurant, pub, fast_food, gym, bakery) = prediction targets.
#   5 always-feature types (supermarket, office, library, university, station) = co-occurrence signals.

def classify_poi_type(row):
    amenity = str(row.get('amenity', '')).lower().strip()
    leisure = str(row.get('leisure', '')).lower().strip()
    shop = str(row.get('shop', '')).lower().strip()
    transport = str(row.get('public_transport', '')).lower().strip()
    office = str(row.get('office', '')).lower().strip()
    # Selectable business types
    if amenity in ('cafe', 'coffee_shop'):        return 'cafe'
    if amenity == 'restaurant':                    return 'restaurant'
    if amenity in ('pub', 'bar'):                  return 'pub'
    if amenity == 'fast_food':                     return 'fast_food'
    if amenity == 'gym' or leisure in ('fitness_centre', 'sports_centre'):
                                                    return 'gym'
    if shop == 'bakery':                           return 'bakery'
    # Always-feature types (not selectable, used as co-occurrence signals)
    if shop == 'supermarket':                      return 'supermarket'
    if (office and office not in ('nan', 'none', '')) or amenity == 'office': return 'office'
    if amenity == 'library':                       return 'library'
    if amenity == 'university':                    return 'university'
    if amenity == 'leisure_centre':                return 'gym'  # merge with gym
    if transport == 'station':                     return 'station'
    return 'other'
# Apply rule-based classification to assign each POI to one of 11 types
pois['poi_type'] = pois.apply(classify_poi_type, axis=1)
# Drop unclassified POIs — these are irrelevant tags not used by the model
pois = pois[pois['poi_type'] != 'other'].copy()  # drop unclassified
# Legacy role column (backward compat for EDA cells)
# WHY LEGACY ROLE MAP: Earlier EDA cells used a simplified 3-class role scheme
#   (Competitor/Synergy/Anchor). The granular poi_type column replaced this for ML,
#   but the role column is retained to avoid breaking EDA cells that reference it.

LEGACY_ROLE_MAP = {
    'cafe': 'Competitor', 'restaurant': 'Synergy', 'pub': 'Synergy',
    'fast_food': 'Other', 'gym': 'Synergy', 'bakery': 'Other',
    'supermarket': 'Other', 'office': 'Synergy', 'library': 'Synergy',
    'university': 'Synergy', 'station': 'Anchor',
}
# Map granular types to legacy 3-class roles for backward-compatible EDA cells
pois['role'] = pois['poi_type'].map(LEGACY_ROLE_MAP).fillna('Other')
print(f"\nTotal POIs fetched (deduplicated): {len(pois)}")
print("\nPOIs by granular type:")
# Summary statistics: verify POI distribution across types
print(pois['poi_type'].value_counts().to_string())
print("\nPOIs by legacy role:")
print(pois['role'].value_counts().to_string())

In [ ]:
# ============================================================
# --- Per-Hex POI Counts and Competition Density ---
# WHY fill_value=0 NOT NaN: A missing column means zero POIs of that type, not
#   missing data. 0 preserves the semantics (no gyms = opportunity). NaN would
#   require explicit handling in XGBoost.

# SECTION 1.4: Spatial Join — Individual POI Counts + Competition Density (k=2)
# ============================================================

# --- Step 1: Per-hex counts for all 11 types ---
pois_in_hex = gpd.sjoin(pois, h3_grid[['h3_index', 'geometry']], how='inner', predicate='within')

type_counts = pois_in_hex.groupby(['h3_index', 'poi_type']).size().unstack(fill_value=0)

# Ensure all type columns exist (some boroughs may lack certain types)
for t in ALL_POI_TYPES:
    if t not in type_counts.columns:
        type_counts[t] = 0

# Rename to n_{type} convention
type_counts = type_counts.rename(columns={t: f'n_{t}' for t in ALL_POI_TYPES})
h3_grid = h3_grid.merge(type_counts[[f'n_{t}' for t in ALL_POI_TYPES]],
                         on='h3_index', how='left')
for t in ALL_POI_TYPES:
    h3_grid[f'n_{t}'] = h3_grid[f'n_{t}'].fillna(0).astype(int)

# --- Step 2: Competition density (k=2 ring neighbors, excluding self) ---
# For each selectable business type, count same-type in the ~18 hexes within ~350m
hex_set = set(h3_grid['h3_index'])
h3_lookup = h3_grid.set_index('h3_index')

print("Computing competition density (k=2 ring, ~350m radius)...")
for biz_key in BUSINESS_TYPES:
    col_self = f'n_{biz_key}'
    col_nearby = f'nearby_{biz_key}'
    nearby_vals = []
    for h3_id in h3_grid['h3_index']:
        # WHY k=2 RING (~350m): Covers the competitive catchment area -- customers within
        #   350m can walk to a competitor as easily as to the target location. k=1 (174m)
        #   is too narrow (misses POIs near open spaces); k=3 (525m) dilutes the signal
        #   by including establishments that do not compete for the same foot traffic.

        neighbors = [nb for nb in h3.grid_disk(h3_id, 2)
                     if nb != h3_id and nb in hex_set]
        total = int(h3_lookup.loc[neighbors, col_self].sum()) if neighbors else 0
        nearby_vals.append(total)
    h3_grid[col_nearby] = nearby_vals
    print(f"  nearby_{biz_key}: mean={np.mean(nearby_vals):.2f}, max={max(nearby_vals)}")

# --- Step 3: Legacy aggregate columns (backward compat for EDA cells) ---
h3_grid['n_competitors'] = h3_grid['n_cafe']
h3_grid['n_synergy'] = (h3_grid['n_gym'] + h3_grid['n_university'] +
                         h3_grid['n_office'] + h3_grid['n_library'] +
                         h3_grid['n_restaurant'] + h3_grid['n_pub'])
h3_grid['n_anchors'] = h3_grid['n_station']
h3_grid['n_other'] = h3_grid['n_bakery'] + h3_grid['n_supermarket'] + h3_grid['n_fast_food']

print(f"\nIndividual POI counts per hex:")
for t in ALL_POI_TYPES:
    total = h3_grid[f'n_{t}'].sum()
    has = (h3_grid[f'n_{t}'] >= 1).sum()
    print(f"  n_{t:12s}: {total:6d} total, {has:5d} hexes with >=1")
print(f"\nLegacy: Competitors={h3_grid['n_competitors'].sum()} | "
      f"Synergy={h3_grid['n_synergy'].sum()} | Anchors={h3_grid['n_anchors'].sum()}")


In [ ]:
# ============================================================
# --- ONS Census 2021 Demographic Enrichment ---
# WHY POPULATION-WEIGHTED MEAN: Multiple census Output Areas (OAs) may fall within
#   a single H3 hex. A simple mean would treat a tiny OA (50 people) equally to a
#   large OA (800 people). Population-weighted averaging produces a demographically
#   accurate aggregate for each hexagon.

# SECTION 1.5: Demographics — ONS Census Enrichment
# ============================================================
# ---- Census data for Greater London (~21K Output Areas) ----
# Downloaded from EDINA Digimap.
CENSUS_ECON_PATH = 'Extra Data Full london/ons-economic-ew-2021_6313715/ons-economic-ew-2021.csv'
CENSUS_AGE_PATH  = 'Extra Data Full london/ons-age-ew-2021_6313714/ons-age-ew-2021.csv'
CENSUS_QUAL_PATH = 'Extra Data Qualifications/ons-qualifications-ew-2021_6313722/ons-qualifications-ew-2021.csv'

# Load all 3 census CSVs
econ_df = pd.read_csv(CENSUS_ECON_PATH)
age_df  = pd.read_csv(CENSUS_AGE_PATH)
qual_df = pd.read_csv(CENSUS_QUAL_PATH)

print(f"Census rows loaded — Economic: {len(econ_df)}, Age: {len(age_df)}, Qualifications: {len(qual_df)}")

# Merge on geog_code
census = econ_df[['geog_code', 'centroid_x', 'centroid_y', 'denom_total',
                   'employed_total_perc', 'retired_perc', 'unemployed_perc']].copy()
census = census.merge(
    age_df[['geog_code', 'age_16_to_34_perc', 'age_65_plus_perc']], on='geog_code', how='left'
)
census = census.merge(
    qual_df[['geog_code', 'level4_perc', 'no_qualifications_perc']], on='geog_code', how='left'
)

# Convert to GeoDataFrame (centroids are already BNG EPSG:27700)
geometry = [Point(xy) for xy in zip(census['centroid_x'], census['centroid_y'])]
census_gdf = gpd.GeoDataFrame(census, geometry=geometry, crs='EPSG:27700')
census_gdf = census_gdf.to_crs(epsg=4326)  # match h3_grid

print(f"Census Output Areas with geometry: {len(census_gdf)}")

# Spatial join: nearest census OA centroid -> hex
census_in_hex = gpd.sjoin_nearest(
    census_gdf, h3_grid[['h3_index', 'geometry']],
    # max_distance=0.005 degrees (~555m at London latitudes): safety guard preventing
    #   census OAs from being assigned to hexagons more than ~0.5km away.

    how='left', max_distance=0.005
)

# Population-weighted mean of demographics per hex
demo_cols = ['employed_total_perc', 'age_16_to_34_perc', 'level4_perc',
             'retired_perc', 'no_qualifications_perc']

def weighted_mean(group, col):
    w = group['denom_total']
    return (group[col] * w).sum() / w.sum() if w.sum() > 0 else 0

hex_demos = []
for h3_id, group in census_in_hex.groupby('h3_index'):
    row = {'h3_index': h3_id}
    for col in demo_cols:
        row[col] = weighted_mean(group, col)
    hex_demos.append(row)

demo_df = pd.DataFrame(hex_demos)
h3_grid = h3_grid.merge(demo_df, on='h3_index', how='left')
for col in demo_cols:
    h3_grid[col] = h3_grid[col].fillna(0)

print(f"\nCensus demographics joined. Coverage: {len(demo_df)}/{len(h3_grid)} hexes")
print(f"  Mean Level 4%: {h3_grid['level4_perc'].mean():.1f}%")
print(f"  Mean Age 16-34%: {h3_grid['age_16_to_34_perc'].mean():.1f}%")
print(f"  Mean Employment%: {h3_grid['employed_total_perc'].mean():.1f}%")

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 1.6: Graph Centrality Features</h2>
    <p>We construct the H3 adjacency graph and compute six centrality metrics per hexagon:</p>
    <ul>
        <li><strong>Degree Centrality</strong>: $C_D(v) = \frac{\deg(v)}{N-1}$ — connectivity (number of neighbours).</li>
        <li><strong>Betweenness Centrality</strong>: $C_B(v) = \sum_{s \neq v \neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}$ — bridging position on shortest paths.</li>
        <li><strong>Closeness Centrality</strong>: $C_C(v) = \frac{N-1}{\sum_{u} d(v,u)}$ — accessibility to all other hexes.</li>
        <li><strong>Clustering Coefficient</strong>: $C_{clust}(v) = \frac{2 T(v)}{\deg(v)(\deg(v)-1)}$ — neighbourhood cohesion.</li>
        <li><strong>Eigenvector Centrality</strong>: $\mathbf{A}\mathbf{x} = \lambda_1 \mathbf{x}$ — a node is important if its neighbours are important (dominant eigenvector of adjacency matrix). Captures <em>prestige</em>: being connected to well-connected hexes.</li>
        <li><strong>PageRank</strong>: $PR(v) = \frac{1-d}{N} + d \sum_{u \in \mathcal{N}_{in}(v)} \frac{PR(u)}{|\mathcal{N}_{out}(u)|}$ — damped random-walk probability. Google's refinement of eigenvector centrality, robust to graph structure irregularities.</li>
    </ul>
    <p>These capture a hexagon's <em>structural role</em> in the urban network — independent of its content.</p>
</div>

In [ ]:
# ============================================================
# --- Graph Centrality Features (6 Metrics per Hexagon) ---
# WHY UNDIRECTED GRAPH: H3 adjacency is symmetric (if A neighbours B, B neighbours A).
#   Pedestrian movement is bidirectional. Undirected uses half the memory of DiGraph.
# WHY 6 METRICS: degree (local connectivity), betweenness (bridge locations),
#   closeness (accessibility), eigenvector (influential neighbours), PageRank
#   (weighted influence with damping), clustering (local density of connections).
#   Each captures a different aspect of spatial network position.

# SECTION 1.6: Graph Centrality Features
# ============================================================
G = nx.Graph()
hex_set = set(h3_grid['h3_index'])

for h3_id in hex_set:
    G.add_node(h3_id)

for h3_id in hex_set:
    for nb in h3.grid_disk(h3_id, 1):
        if nb != h3_id and nb in hex_set:
            G.add_edge(h3_id, nb)

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
print(f"Graph: {n_nodes} nodes, {n_edges} edges")

# Scale-aware betweenness: approximate for large graphs (>1000 nodes)
# Exact betweenness is O(V*E) — infeasible for ~55K nodes (~10-50 hours)
# k-sample approximation: O(k*E) — ~100x faster, preserves ranking
APPROX_K = min(500, n_nodes)
use_approx = n_nodes > 1000

# Degree centrality — O(V+E), always exact
h3_grid['degree_centrality'] = h3_grid['h3_index'].map(nx.degree_centrality(G))
print("  Degree centrality: done")

# Betweenness centrality — approximate if graph is large
if use_approx:
    print(f"  Betweenness centrality: approximating with k={APPROX_K} samples...")
    h3_grid['betweenness_centrality'] = h3_grid['h3_index'].map(
        nx.betweenness_centrality(G, k=APPROX_K)
    )
else:
    h3_grid['betweenness_centrality'] = h3_grid['h3_index'].map(
        nx.betweenness_centrality(G)
    )
print("  Betweenness centrality: done")

# Closeness centrality — O(V*(V+E)), feasible up to ~60K nodes
h3_grid['closeness_centrality'] = h3_grid['h3_index'].map(nx.closeness_centrality(G))
print("  Closeness centrality: done")

# Clustering coefficient — O(V*d²), efficient for sparse graphs
h3_grid['clustering_coeff'] = h3_grid['h3_index'].map(nx.clustering(G))
print("  Clustering coefficient: done")

# Eigenvector Centrality: dominant eigenvector of adjacency matrix A*x = λ₁*x
# Captures "prestige" — a hex is important if its neighbours are important
# Computed per connected component with fallback chain for numerical stability
eig_dict = {}
eig_fallbacks = 0
for comp in nx.connected_components(G):
    subG = G.subgraph(comp)
    if len(comp) <= 2:
        for node in comp:
            eig_dict[node] = 1.0 / len(comp)
    else:
        try:
            eig_dict.update(nx.eigenvector_centrality_numpy(subG))
        except Exception:
            try:
                eig_dict.update(nx.eigenvector_centrality(subG, max_iter=5000, tol=1e-04))
            except Exception:
                eig_fallbacks += 1
                deg = nx.degree_centrality(subG)
                eig_dict.update(deg)
h3_grid['eigenvector_centrality'] = h3_grid['h3_index'].map(eig_dict)
n_comp = nx.number_connected_components(G)
msg = f"  Eigenvector centrality: done ({n_comp} components)"
if eig_fallbacks:
    msg += f" [{eig_fallbacks} used degree-centrality fallback]"
print(msg)

# PageRank: damped random-walk probability (Google's refinement of eigenvector centrality)
# More robust than raw eigenvector centrality for irregular graph structures
h3_grid['pagerank'] = h3_grid['h3_index'].map(
    nx.pagerank(G, alpha=0.85)
)
print("  PageRank: done")

print(f"\nAll 6 centrality metrics computed {'(betweenness approximated)' if use_approx else '(all exact)'}.")
print(f"Mean degree centrality:      {h3_grid['degree_centrality'].mean():.4f}")
print(f"Mean betweenness centrality: {h3_grid['betweenness_centrality'].mean():.4f}")
print(f"Mean eigenvector centrality: {h3_grid['eigenvector_centrality'].mean():.4f}")
print(f"Mean PageRank:               {h3_grid['pagerank'].mean():.6f}")

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Concept: Spatial Graphs and Centrality</h2>
    <p>We model Greater London as a <strong>graph</strong> where:</p>
    <ul>
        <li><strong>Nodes</strong> = H3 hexagons (carrying population, demographics, and POI counts).</li>
        <li><strong>Edges</strong> = adjacency links between touching hexagons, weighted by Inverse Distance: $w_{uv} = \frac{1}{d(u,v) + 1}$</li>
    </ul>
    <p>From this graph we compute six <strong>centrality metrics</strong> that quantify each hexagon's structural position:</p>
    <table style="width:100%; border-collapse: collapse; margin: 15px 0; font-size: 0.95em;">
        <tr style="background: #2e86c1; color: white;">
            <th style="padding: 8px;">Metric</th>
            <th style="padding: 8px;">Intuition</th>
            <th style="padding: 8px;">Business Meaning</th>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>Degree</strong></td>
            <td>How many neighbours?</td>
            <td>Interior vs. boundary &mdash; interior hexes have 6 neighbours</td>
        </tr>
        <tr style="background: #eaf2f8;">
            <td style="padding: 8px;"><strong>Betweenness</strong></td>
            <td>How often on shortest paths?</td>
            <td>Transit corridors &mdash; high foot traffic pass-through</td>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>Closeness</strong></td>
            <td>Average distance to all others?</td>
            <td>Central vs. peripheral location within the study area</td>
        </tr>
        <tr style="background: #eaf2f8;">
            <td style="padding: 8px;"><strong>Clustering</strong></td>
            <td>How interconnected are neighbours?</td>
            <td>Neighbourhood cohesion &mdash; tightly knit areas</td>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>Eigenvector</strong></td>
            <td>Connected to <em>important</em> nodes? ($\mathbf{A}\mathbf{x}=\lambda_1\mathbf{x}$)</td>
            <td>Prestige &mdash; hexes in high-connectivity clusters (commercial hubs)</td>
        </tr>
        <tr style="background: #eaf2f8;">
            <td style="padding: 8px;"><strong>PageRank</strong></td>
            <td>Damped random-walk probability</td>
            <td>Organic reachability &mdash; how likely a random walker arrives here</td>
        </tr>
    </table>
    <p>These metrics feed directly into the ML pipeline as features for the binary classification model.
    The spectral metrics (eigenvector centrality, PageRank) add mathematical depth by encoding
    <em>recursive neighbourhood quality.</p>
    <div style="background: #eaf2f8; padding: 12px; border-radius: 5px; border-left: 4px solid #2e86c1; margin-top: 10px;">
        <strong>Why include both Eigenvector Centrality and PageRank?</strong>
        Eigenvector centrality solves <em>Ax = &lambda;x</em> and assumes all influence flows symmetrically. PageRank adds a <strong>damping factor</strong> (&alpha; = 0.85): at each step, there is a 15% chance the random walker teleports to a random node. This handles <em>dangling nodes</em> (hexagons at the study area boundary with few neighbours) that would otherwise accumulate spurious eigenvector importance. In practice, PageRank is more stable on disconnected or near-disconnected components (e.g., river-separated boroughs), while eigenvector centrality captures pure topological prestige.
    </div>
</div>

In [ ]:
# ============================================================
# CHECKPOINT: Save Intermediate Files
# ============================================================
# These intermediate outputs support the Streamlit dashboard
# and provide reproducibility checkpoints.

# WHY SAVE IN EPSG:27700 (BNG): All downstream distance/area computations must
#   use a projected CRS. Storing in BNG prevents accidental Euclidean distance
#   calculations in EPSG:4326 (which gives distances in degrees, not metres).

h3_grid_bng = h3_grid.to_crs(epsg=27700)
h3_grid_bng.to_parquet('data/outputs/london_h3_grid.parquet')

keep_cols = ['geometry', 'role', 'amenity', 'leisure', 'public_transport', 'name', 'shop']
pois_save = pois[[c for c in keep_cols if c in pois.columns]].copy()
pois_save.to_file('data/processed/london_pois.geojson', driver='GeoJSON')

# Save census as GeoJSON for reference
census_gdf.to_file('data/processed/london_census_oa.geojson', driver='GeoJSON')

print(f"Saved: data/outputs/london_h3_grid.parquet ({len(h3_grid)} hexagons, {h3_grid['borough'].nunique()} boroughs)")
print(f"Saved: data/processed/london_pois.geojson ({len(pois_save)} POIs)")
print(f"Saved: data/processed/london_census_oa.geojson ({len(census_gdf)} OAs)")
print("Checkpoint complete.")

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 2: Target Variable — Binary Coffee Shop Presence</h2>
    <p>We define the prediction target as:</p>
    <p style="text-align: center; font-size: 1.2em;">$$y_H = \begin{cases} 1 & \text{if hexagon } H \text{ contains } \geq 1 \text{ cafe or coffee\_shop} \\ 0 & \text{otherwise} \end{cases}$$</p>

    <div style="background: #f2f3f4; padding: 12px; border-radius: 5px; border-left: 4px solid #5d6d7e; margin-top: 10px;">
        <strong>Population Filter:</strong> Before building the feature matrix, we remove all hexagons with
        <code>population == 0</code>. These correspond to parks (Regent's Park, Hampstead Heath, Richmond Park),
        cemeteries, railway land, and reservoirs — locations where a coffee shop cannot feasibly operate.
        Retaining them would inflate True Negatives and produce misleading recommendations in uninhabitable areas.
    </div>

    <p><strong>Class imbalance is expected</strong>: most hexagons will not contain a coffee shop. We handle this with <code>class_weight='balanced'</code> (LR, RF) and <code>scale_pos_weight</code> (XGBoost).</p>

    <div style="background: #eaf2f8; padding: 12px; border-radius: 5px; border-left: 4px solid #2e86c1; margin-top: 10px;">
        <strong>Success Metrics &amp; Constraints</strong>
        <ul style="margin: 8px 0;">
            <li><strong>Primary metric</strong>: ROC-AUC &mdash; threshold-invariant measure of discrimination. Chosen over accuracy because class imbalance makes accuracy misleading (a naive all-zero classifier achieves ~80% accuracy).</li>
            <li><strong>Secondary</strong>: Precision at top-20 &mdash; are the top-ranked recommendations credible to an investor?</li>
            <li><strong>Tertiary</strong>: Recall &mdash; do we miss viable locations? Low recall means leaving revenue on the table.</li>
        </ul>
        <strong>Scope constraints:</strong>
        <ul style="margin: 8px 0;">
            <li>Covers all 33 London boroughs &mdash; model may not generalise to other UK cities without retraining.</li>
            <li>Static temporal snapshot: ONS Census 2021, OSM data from the notebook run date. No longitudinal dynamics.</li>
            <li>Binary target (presence/absence) &mdash; no ground-truth revenue or profitability data available.</li>
            <li>No commercial rent, planning permission, or lease availability data &mdash; the model identifies <em>demand-side</em> opportunity only.</li>
        </ul>
    </div>

        <div style="background: #d4efdf; padding: 12px; border-radius: 5px; border-left: 4px solid #27ae60; margin-top: 10px;">
        <strong>Why binary classification, not count regression?</strong>
        A count model (Poisson, negative binomial) would predict <em>how many</em> coffee shops a hexagon should contain. However: (a) most hexagons have 0 or 1 shop, making the count distribution severely zero-inflated; (b) "how many" conflates demand with supply-side decisions (franchise density, lease availability); (c) the business question is "where to open <em>a</em> shop" &mdash; a binary yes/no. Classification directly answers this question and avoids modelling nuisance variation in count data.
    </div>

<div style="background: #fff3cd; padding: 12px; border-radius: 5px; border-left: 4px solid #ffc107; margin-top: 10px;">
        <strong>Leakage Warning:</strong> The <code>n_competitors</code> column counts cafes/coffee shops per hex &mdash; this directly encodes the target.
        It <strong>must be excluded</strong> from the feature matrix <code>X</code>. We retain <code>n_synergy</code> and <code>n_anchors</code>
        as legitimate predictors (synergy nodes attract coffee shops, not vice versa).
    </div>
</div>

In [ ]:
# ============================================================
# --- Target Variable Construction & Feature Matrix Assembly ---
# WHY REMOVE ZERO-POPULATION HEXES: Hexagons covering parks (Hyde Park, Regent's
#   Park) and the Thames have zero LandScan population. Including them as negatives
#   would teach the model a trivial rule: "no people = no cafe". Removing these
#   uninhabitable hexes prevents learning spurious population-exclusion rules.
# FEATURE COUNT: 6 demographic + 6 centrality + 10 co-occurrence (11 types minus
#   self) + 1 competition density = 23 features per business type.

# SECTION 2: Target Definition & Feature Matrices (Multi-Type)
# ============================================================

# --- Population filter: remove uninhabitable hexagons ---
n_before = len(h3_grid)
n_zero_pop = (h3_grid['population'] == 0).sum()
h3_grid = h3_grid[h3_grid['population'] > 0].reset_index(drop=True)
n_after = len(h3_grid)

print(f"Population filter: {n_before} -> {n_after} hexagons "
      f"({n_zero_pop} zero-population hexes removed)")

# --- Define targets for all 6 business types ---
print("\nTarget distributions (after population filter):")
for biz_key in BUSINESS_TYPES:
    h3_grid[f'has_{biz_key}'] = (h3_grid[f'n_{biz_key}'] >= 1).astype(int)
    n_pos = h3_grid[f'has_{biz_key}'].sum()
    rate = h3_grid[f'has_{biz_key}'].mean()
    print(f"  has_{biz_key:12s}: {n_pos:5d} positive ({rate:.1%})")

# Legacy alias
h3_grid['has_coffee_shop'] = h3_grid['has_cafe']

# --- Shared (non-POI) feature columns ---
DEMOGRAPHIC_COLS = ['population', 'employed_total_perc', 'age_16_to_34_perc',
                    'level4_perc', 'retired_perc', 'no_qualifications_perc']
CENTRALITY_COLS = ['degree_centrality', 'betweenness_centrality',
                   'closeness_centrality', 'clustering_coeff',
                   'eigenvector_centrality', 'pagerank']

# --- Build per-type feature matrices ---
# For type T: demographics(6) + centrality(6) + n_{all other types}(10)
#             + nearby_{T}(1) = 23 features
# LEAKAGE GUARD: n_{T} is NEVER included when predicting T

type_datasets = {}  # {biz_key: (X, y, feature_cols)}

for biz_key in BUSINESS_TYPES:
    # Co-occurrence features: all POI type counts EXCEPT the target type
    poi_feature_cols = [f'n_{t}' for t in ALL_POI_TYPES if t != biz_key]
    # Competition density: same-type count in adjacent hexes
    competition_col = [f'nearby_{biz_key}']

    feature_cols = DEMOGRAPHIC_COLS + CENTRALITY_COLS + poi_feature_cols + competition_col
    X_t = h3_grid[feature_cols].copy()
    y_t = h3_grid[f'has_{biz_key}'].copy()

    # WHY ASSERTION NOT JUST A COMMENT: If n_cafe were accidentally included in the
    #   cafe feature matrix, the model would achieve near-perfect AUC (the feature
    #   directly encodes the target). A hard runtime assertion catches this instantly.

    # Leakage assertion
    assert f'n_{biz_key}' not in X_t.columns, \
        f"LEAKAGE: n_{biz_key} found in feature set for {biz_key}!"

    type_datasets[biz_key] = (X_t, y_t, feature_cols)

print(f"\nFeature matrices built for {len(type_datasets)} business types.")
for biz_key, (X_t, y_t, feat_cols) in type_datasets.items():
    print(f"  {biz_key:12s}: X={X_t.shape}, positive_rate={y_t.mean():.1%}, "
          f"features={len(feat_cols)}")

# Legacy aliases for EDA cells (point to cafe — primary type)
X, y = type_datasets['cafe'][0], type_datasets['cafe'][1]
FEATURE_COLS = type_datasets['cafe'][2]
TARGET_COL = 'has_cafe'
imbalance_ratio = (y == 0).sum() / (y == 1).sum()

# Leakage assertion
assert 'n_cafe' not in X.columns, "LEAKAGE: n_cafe found in feature set!"

print(f"\nPrimary type (cafe): X={X.shape}, imbalance_ratio={imbalance_ratio:.1f}:1")
print(f"Features ({len(FEATURE_COLS)}):")
for i, f in enumerate(FEATURE_COLS):
    print(f"  {i+1:2d}. {f}")
print(f"\nMissing values per feature:")
print(X.isnull().sum().to_string())


<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 3: Exploratory Data Analysis & Leakage Audit</h2>
    <p>Before modelling, we verify:</p>
    <ol>
        <li><strong>Feature distributions</strong>: Zero variance, extreme skew, or missing values?</li>
        <li><strong>Multicollinearity</strong>: Variance Inflation Factors (VIF) — flag features with VIF &gt; 10.</li>
        <li><strong>Spatial autocorrelation</strong>: Visual confirmation that the target is spatially clustered (motivating Spatial CV).</li>
    </ol>
</div>

In [ ]:
# ============================================================
# --- Missingness Audit & Class Balance Check ---
# WHY 5% THRESHOLD: Industry convention. Features below 5% missing can be safely
#   zero-imputed without materially affecting model performance.
# WHY SHOW NAIVE ACCURACY: If 88% of hexes have y=0, a trivial all-negative
#   classifier achieves 88% accuracy. This motivates ROC-AUC as the primary
#   metric (naive AUC = 0.5 regardless of imbalance).

# SECTION 3-pre: Missingness Audit & Class Balance
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Missingness summary ---
missing = X.isnull().sum()
missing_pct = (X.isnull().sum() / len(X) * 100)
miss_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct}).sort_values('Count', ascending=False)

axes[0].barh(miss_df.index, miss_df['Percent'], color='#e74c3c')
axes[0].set_xlabel('Missing (%)')
axes[0].set_title('Feature Missingness', fontsize=12, fontweight='bold')
axes[0].axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='5% threshold')
axes[0].legend()

# --- Right: Class balance ---
class_counts = y.value_counts().sort_index()
bars = axes[1].bar(
    ['No Coffee (0)', 'Has Coffee (1)'],
    class_counts.values,
    color=['#3498db', '#e74c3c']
)
for bar, val in zip(bars, class_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', fontweight='bold')
axes[1].set_title('Target Class Distribution', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Hexagons')

imbalance = class_counts[0] / class_counts[1]
axes[1].annotate(f'Imbalance ratio: {imbalance:.1f}:1',
                 xy=(0.5, 0.85), xycoords='axes fraction',
                 ha='center', fontsize=11, fontweight='bold',
                 bbox=dict(boxstyle='round', facecolor='#fff3cd', edgecolor='#ffc107'))

plt.tight_layout()
fig.savefig('docs/assets/missingness_class_balance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Missingness summary: {(missing > 0).sum()} features have missing values")
print(f"Class balance: {class_counts[0]} negative, {class_counts[1]} positive "
      f"(ratio {imbalance:.1f}:1)")
print(f"  -> A naive all-zero classifier achieves {class_counts[0]/len(y)*100:.1f}% accuracy")
print(f"  -> This motivates ROC-AUC as the primary metric (threshold-invariant)")

In [ ]:
# ============================================================
# --- Correlation Heatmap ---
# WHY UPPER TRIANGLE MASK: The matrix is symmetric (corr(A,B) == corr(B,A)).
#   Masking the upper half removes redundancy, improving readability.
# WHY RdBu_r COLORMAP: Red=positive, blue=negative, white=zero. The standard
#   academic choice for correlation matrices (center=0 ensures symmetric coloring).

# SECTION 3a: Correlation Heatmap
# ============================================================
fig, ax = plt.subplots(figsize=(12, 10))

corr = X.join(y).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax
)
ax.set_title('Feature Correlation Matrix (incl. Target)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/outputs/eda_correlations.png', dpi=150, bbox_inches='tight')
fig.savefig('docs/assets/correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Flag high correlations with target
target_corr = corr[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False)
print(f"Feature correlations with target ({TARGET_COL}):")
print(target_corr.to_string())


In [ ]:
# ============================================================
# --- Feature Distributions by Target Class ---
# WHY BOXPLOTS NOT HISTOGRAMS: Boxplots compare two distributions (y=0 vs y=1)
#   on the same axis, showing median, IQR, and outliers simultaneously.
#   Overlaid histograms require transparency tuning and are harder to compare.
# WHY THESE FEATURES: One from each modality -- population (footfall), level4_perc
#   (affluence/education), age_16_to_34 (coffee demographic), betweenness (accessibility),
#   n_restaurant (synergy), nearby_cafe (competition). Balanced EDA overview.

# SECTION 3b: Feature Distributions by Target Class
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
key_features = ['population', 'level4_perc', 'age_16_to_34_perc',
                'betweenness_centrality', 'n_restaurant', 'nearby_cafe']

available = [f for f in key_features if f in X.columns]
for ax, feat in zip(axes.ravel(), available):
    plot_df = X[[feat]].copy()
    plot_df['target'] = y.values
    sns.boxplot(data=plot_df, x='target', y=feat, ax=ax,
                hue='target', palette={0: '#3498db', 1: '#e74c3c'}, legend=False)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('has_cafe')

plt.suptitle('Feature Distributions by Target Class (Cafe)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/outputs/eda_boxplots.png', dpi=150, bbox_inches='tight')
fig.savefig('docs/assets/feature_distributions.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# ============================================================
# --- Variance Inflation Factor (Multicollinearity Check) ---
# WHY VIF: Measures how much a feature's variance is inflated by correlation with
#   other features. VIF > 10 = strong multicollinearity, which destabilises LR
#   coefficients. XGBoost is more robust but we check for the LR baseline.
# WHY REMOVE ZERO-VARIANCE FIRST: VIF inverts the covariance matrix. Constant
#   columns produce a singular matrix causing LinAlgError.

# SECTION 3c: Variance Inflation Factor (Multicollinearity)
# ============================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF requires no NaN and no constant columns
X_vif = X.fillna(0).copy()
# Remove zero-variance columns for VIF calculation
X_vif = X_vif.loc[:, X_vif.std() > 0]

vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)

print("Variance Inflation Factors:")
print(vif_data.to_string(index=False))
print(f"\nFeatures with VIF > 10 (potential multicollinearity): "
      f"{list(vif_data[vif_data['VIF'] > 10]['Feature'])}")

<div style="margin-top: 20px; padding: 15px; background-color: #fdf2e9; border-radius: 8px; border-left: 5px solid #d35400;">
    <h3 style="color: #d35400;">Leakage Audit</h3>
    <p>Before proceeding to modelling, we formally document potential leakage vectors and our mitigations:</p>
    <table style="width:100%; border-collapse: collapse; margin: 10px 0; font-size: 0.9em;">
        <tr style="background: #d35400; color: white;">
            <th style="padding: 8px;">Risk</th>
            <th style="padding: 8px;">Feature(s)</th>
            <th style="padding: 8px;">Assessment</th>
            <th style="padding: 8px;">Mitigation</th>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>Direct target encoding</strong></td>
            <td><code>n_competitors</code></td>
            <td style="color: #c0392b;">HIGH &mdash; counts cafes/coffee shops, which <em>are</em> the target</td>
            <td>Excluded from <code>X</code>. Enforced by runtime assertion.</td>
        </tr>
        <tr style="background: #fef9f4;">
            <td style="padding: 8px;"><strong>Reverse causation</strong></td>
            <td><code>n_synergy</code>, <code>n_anchors</code></td>
            <td style="color: #e67e22;">MODERATE &mdash; do coffee shops attract gyms, or do gyms attract coffee shops?</td>
            <td>Retained. Causal direction is predominantly synergy &rarr; coffee (gyms/offices pre-date most cafes). Acknowledged as a limitation in the Model Card.</td>
        </tr>
        <tr>
            <td style="padding: 8px;"><strong>Temporal leakage</strong></td>
            <td>All OSM features</td>
            <td style="color: #e67e22;">MODERATE &mdash; OSM snapshot is contemporaneous with target; no temporal split</td>
            <td>Accepted. This is a cross-sectional study. Acknowledged in scope constraints.</td>
        </tr>
        <tr style="background: #fef9f4;">
            <td style="padding: 8px;"><strong>Spatial leakage</strong></td>
            <td>All features (via autocorrelation)</td>
            <td style="color: #c0392b;">HIGH &mdash; adjacent hexes share similar feature values (Tobler's Law)</td>
            <td>Controlled by Spatial Block CV (Section 4). Adjacent hexes always in same fold.</td>
        </tr>
    </table>
    <div style="background: #eaf2f8; padding: 12px; border-radius: 5px; border-left: 4px solid #2e86c1; margin-top: 10px;">
        <strong>Note on <code>nearby_cafe</code>:</strong> This feature counts cafes in the k=2 ring <em>surrounding</em> the target hexagon, <strong>not</strong> within the hexagon itself. It captures the competitive landscape (saturation vs. opportunity) without encoding the target variable. A hexagon with y=1 (has a cafe) may have nearby_cafe=0 (no competitors nearby) or nearby_cafe=5 (saturated area). The feature provides legitimate competitive-environment signal, not direct target leakage.
    </div>

</div>

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 4: Spatial Cross-Validation Strategy</h2>
    <p><strong>Problem:</strong> Standard <code>KFold</code> randomly assigns hexagons to train/test folds.
    Adjacent hexes share similar features due to spatial autocorrelation
    (<em>Tobler's First Law of Geography</em>: "near things are more related than distant things").
    Random splits leak spatial signal, inflating performance metrics by 5–15%.</p>
    
    <p><strong>Solution:</strong> <strong>Spatial Block Cross-Validation</strong> using the H3 hierarchy.
    We group Resolution-9 hexagons by their Resolution-7 parent cell (~460 m blocks).
    All hexes sharing a parent are assigned to the same fold, preserving spatial integrity.</p>
    
    <p style="text-align: center; font-size: 1.1em;">$$\text{fold}(H) = \text{GreedyAssign}\big(\text{h3\_cell\_to\_parent}(H, \text{res}=7)\big)$$</p>

    <p><strong>Greedy class balancing:</strong> Rather than na&iuml;ve modular assignment ($\text{block}_i \bmod k$),
    we sort spatial blocks by their positive class rate and greedily assign each block to the fold
    with the fewest positives so far. This guarantees both classes appear in every fold &mdash;
    preventing degenerate folds that would produce <code>NaN</code> AUC scores.</p>

    <div style="background: #fadbd8; padding: 12px; border-radius: 5px; border-left: 4px solid #c0392b; margin-top: 10px;">
        <strong>Block size exceeds feature leakage radius:</strong> Resolution-7 parent cells have ~2.2 km diameter. The competition density feature (<code>nearby_cafe</code>) uses a k=2 H3 ring, which at Resolution 9 spans ~350 m. Since 2.2 km &gt;&gt; 350 m, hexagons in different CV folds cannot share competition density information. This is the key design constraint: the spatial block must be larger than the maximum feature interaction radius to prevent information leakage across folds.
    </div>
</div>

In [ ]:
# ============================================================
# --- Spatial Block Cross-Validation ---
# WHY NOT STANDARD K-FOLD: Standard KFold randomly assigns hexagons to folds.
#   Adjacent hexagons share similar demographics, population, and POI density.
#   If neighbours end up in different folds, the model "sees" the test area
#   during training via these correlated features = spatial data leakage.

# SECTION 4: Spatial Block Cross-Validation (H3 Parent-Cell)
# ============================================================

class SpatialKFold:
    # WHY RESOLUTION 7 PARENTS: Res-7 cells have ~460m edge, grouping ~50 Res-9
    #   children into ~2.2km diameter blocks. This exceeds the k=2 ring radius
    #   (~350m) used for competition density, preventing the nearby_{type} feature
    #   from leaking across the train/test boundary.

    """Spatial block cross-validation using H3 parent-cell partitioning.

    Groups H3 Res-9 hexagons by their parent cell, then assigns
    spatial blocks to folds using a greedy class-balancing strategy.
    Ensures geographically proximate hexagons never appear in both
    train and test simultaneously, while guaranteeing both classes
    appear in every fold.
    """

    def __init__(self, n_splits=5, parent_resolution=7):
        self.n_splits = n_splits
        self.parent_resolution = parent_resolution

    def split(self, X, y=None, groups=None):
        parents = groups.apply(
            # Map each Res-9 hex to its Res-7 parent (~2.2km block) for spatial grouping
            lambda h: h3.cell_to_parent(h, self.parent_resolution)
        )

        # Compute positive rate per parent block for stratified assignment
        block_df = pd.DataFrame({'parent': parents, 'y': y.values})
        block_stats = block_df.groupby('parent')['y'].agg(['sum', 'count'])
        block_stats.columns = ['n_pos', 'n_total']
        # Sort blocks by positive rate descending — greedy balancing
        block_stats['pos_rate'] = block_stats['n_pos'] / block_stats['n_total']
        block_stats = block_stats.sort_values('pos_rate', ascending=False)

        # Greedy assignment: assign each block to the fold with the fewest positives
        fold_pos_counts = np.zeros(self.n_splits)
        parent_to_fold = {}
        for parent_id in block_stats.index:
            # WHY GREEDY NOT ROUND-ROBIN: Simple modular assignment (block_i % k) produces
            #   wildly imbalanced folds when positives (coffee shops) are spatially clustered.
            #   The greedy strategy assigns each block to the most-deficient fold, ensuring
            #   both classes appear in every fold. O(B log B) in number of blocks.

            fold = int(np.argmin(fold_pos_counts))
            parent_to_fold[parent_id] = fold
            fold_pos_counts[fold] += block_stats.loc[parent_id, 'n_pos']

        fold_assignments = parents.map(parent_to_fold)

        for fold_idx in range(self.n_splits):
            test_mask = fold_assignments == fold_idx
            train_idx = np.where(~test_mask)[0]
            test_idx = np.where(test_mask)[0]
            # Yields numpy index arrays (not boolean masks) to match sklearn cross_val_score interface.

            # Yield numpy index arrays (sklearn-compatible train/test split format)
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


spatial_cv = SpatialKFold(n_splits=5, parent_resolution=7)

# Verify fold sizes and class balance
print("Spatial CV Fold Sizes:")
for i, (train_idx, test_idx) in enumerate(
    spatial_cv.split(X, y, groups=h3_grid['h3_index'])
):
    train_pos = y.iloc[train_idx].mean()
    test_pos = y.iloc[test_idx].mean()
    # Auditable output: show fold composition for examiner verification
    print(f"  Fold {i+1}: train={len(train_idx)} ({train_pos:.1%} pos), "
          f"test={len(test_idx)} ({test_pos:.1%} pos)")

# Count spatial blocks
n_blocks = h3_grid['h3_index'].apply(
    lambda h: h3.cell_to_parent(h, 7)
).nunique()
print(f"\nSpatial blocks (Res-7 parents): {n_blocks}")
print("Greedy assignment ensures both classes in every fold.")

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 5: Baseline Model — Logistic Regression</h2>
    <p>Logistic Regression serves as our <strong>interpretability baseline</strong>. Its linear decision boundary provides
    directly readable coefficients — the sign and magnitude of each feature's effect on coffee shop presence.</p>
    <p>We apply <code>StandardScaler</code> (required for LR convergence) and <code>class_weight='balanced'</code>
    to handle the expected class imbalance.</p>
</div>

In [ ]:
# ============================================================
# --- Logistic Regression Baseline ---
# WHY StandardScaler REQUIRED: LR gradient descent convergence depends on feature
#   scale. Unscaled features (population in thousands vs level4_perc in 0-100) cause
#   disproportionate gradient steps. StandardScaler normalises to mean=0, std=1.
# WHY LR AS BASELINE: Its coefficients are directly readable as feature effects,
#   providing an interpretability reference that XGBoost cannot easily match.

# SECTION 5: Baseline — Logistic Regression
# ============================================================
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ))
])

lr_scores = cross_val_score(
    lr_pipe, X, y,
    cv=spatial_cv.split(X, y, groups=h3_grid['h3_index']),
    scoring='roc_auc'
)

print(f"Logistic Regression (Spatial CV):")
print(f"  ROC-AUC: {lr_scores.mean():.3f} \u00b1 {lr_scores.std():.3f}")
print(f"  Per-fold: {[f'{s:.3f}' for s in lr_scores]}")

# Fit on full data for coefficient inspection
lr_pipe.fit(X, y)
lr_coefs = pd.Series(
    lr_pipe.named_steps['model'].coef_[0],
    index=FEATURE_COLS
).sort_values()

print(f"\nLogistic Regression Coefficients (standardised):")
print(lr_coefs.to_string())

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 6: Model Comparison — Random Forest & XGBoost</h2>
    <p>We compare three models spanning the complexity spectrum:</p>
    <table style="width:100%; border-collapse: collapse; margin-top: 10px;">
        <tr style="background: #16213e; color: white;">
            <th style="padding: 8px;">Model</th><th>Type</th><th>Class Imbalance Handling</th><th>Rationale</th>
        </tr>
        <tr><td style="padding: 8px;">Logistic Regression</td><td>Linear</td><td><code>class_weight='balanced'</code></td><td>Interpretable baseline</td></tr>
        <tr style="background: #f8f9fa;"><td style="padding: 8px;">Random Forest</td><td>Bagged ensemble</td><td><code>class_weight='balanced'</code></td><td>Non-linear, robust to outliers</td></tr>
        <tr><td style="padding: 8px;">XGBoost</td><td>Boosted ensemble</td><td><code>scale_pos_weight</code></td><td>State-of-the-art tabular performance</td></tr>
    </table>

    <div style="background: #f5eef8; padding: 12px; border-radius: 5px; border-left: 4px solid #884ea0; margin-top: 10px;">
        <strong>Imbalance handling rationale per model:</strong>
        <ul style="margin: 5px 0;">
            <li><strong>Logistic Regression</strong> (<code>class_weight='balanced'</code>): Inversely weights classes by frequency, equivalent to oversampling the minority class. This adjusts the decision boundary without synthetic data generation.</li>
            <li><strong>Random Forest</strong> (<code>class_weight='balanced'</code>): Same inverse-frequency weighting applied per bootstrap sample. Each tree sees effectively balanced data, reducing the ensemble's bias toward the majority class.</li>
            <li><strong>XGBoost</strong> (<code>scale_pos_weight = neg/pos</code>): Multiplies the gradient contribution of positive samples by the imbalance ratio. This is XGBoost's native mechanism and is more numerically stable than sample weighting for boosted ensembles (Chen &amp; Guestrin, 2016).</li>
        </ul>
    </div>
</div>

In [ ]:
# ============================================================
# --- Model Comparison: Logistic Regression vs Random Forest vs XGBoost ---
# WHY THESE THREE MODELS: LR provides a linear interpretability baseline (readable
#   coefficients). RF captures non-linear interactions via bagging. XGBoost (gradient-
#   boosted trees) is SOTA for structured tabular data -- its sequential boosting on
#   residuals handles spatial heterogeneity better than bagging (Chen & Guestrin 2016).
# WHY ROC-AUC NOT ACCURACY: With ~8:1 class imbalance, a naive all-negative classifier
#   achieves 88% accuracy. AUC is threshold-invariant and rewards models that rank
#   positive hexagons higher than negative ones -- exactly what site selection needs.

# SECTION 6: Model Comparison — LR vs RF vs XGBoost
# ============================================================
imbalance_ratio = (y == 0).sum() / (y == 1).sum()

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        # Linear baseline: interpretable coefficients show feature effects directly
        ('model', LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE  # scales sample weights inversely proportional to class frequency
        ))
    ]),
    # Bagged ensemble: non-linear, robust to outliers, captures interactions
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    # Boosted ensemble: state-of-the-art for tabular data (Chen & Guestrin 2016)
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200,
        # WHY scale_pos_weight: XGBoost's native class weighting. Multiplies the gradient
        #   of positive-class errors by the imbalance ratio, equivalent to oversampling
        #   the minority class during training without actually duplicating samples.

        scale_pos_weight=imbalance_ratio,
        eval_metric='logloss',
        random_state=RANDOM_STATE
    )
}

results = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X, y,
        cv=spatial_cv.split(X, y, groups=h3_grid['h3_index']),
        # ROC-AUC: threshold-invariant; not biased by class imbalance (unlike accuracy)
        scoring='roc_auc'
    )
    results[name] = {
        'mean_auc': scores.mean(),
        'std_auc': scores.std(),
        'scores': scores
    }
    print(f"{name:25s} ROC-AUC = {scores.mean():.3f} \u00b1 {scores.std():.3f}")

# Summary table
results_df = pd.DataFrame({
    'Model': results.keys(),
    'Mean AUC': [r['mean_auc'] for r in results.values()],
    'Std AUC': [r['std_auc'] for r in results.values()]
}).sort_values('Mean AUC', ascending=False)

print(f"\nModel Comparison Summary:")
results_df

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 7: Hyperparameter Tuning</h2>
    <p>We tune the best-performing model using <code>GridSearchCV</code> with our spatial CV splitter.
    The search space covers tree depth, learning rate, subsampling, and column sampling —
    the most impactful XGBoost hyperparameters for tabular classification.</p>
</div>

In [ ]:
# ============================================================
# --- XGBoost Hyperparameter Tuning via GridSearchCV ---
# WHY THESE RANGES (Chen & Guestrin 2016 standard grid):
#   n_estimators (100-300): more trees = stability, diminishing returns past ~200
#   max_depth (3-7): shallow=underfit, deep=overfit for tabular spatial data
#   learning_rate (0.01-0.1): lower LR + more trees is generally better
#   subsample (0.7-0.9): stochastic gradient boosting reduces variance
#   colsample_bytree (0.7-1.0): feature subsampling adds regularisation

# SECTION 7: Hyperparameter Tuning (XGBoost — Cafe Primary Type)
# ============================================================
# Full GridSearchCV on cafe (primary type), then reuse params for all 6 types.
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 1.0]
}

xgb_base = xgb.XGBClassifier(
    scale_pos_weight=imbalance_ratio,
    eval_metric='logloss',
    random_state=RANDOM_STATE
)

# Pre-compute spatial CV splits for GridSearchCV
# WHY list() CONVERSION: GridSearchCV iterates cv multiple times (once per param
#   combination). A generator would be exhausted after the first pass. Converting
#   to a list materialises all fold splits in memory for reuse.

cv_splits = list(spatial_cv.split(X, y, groups=h3_grid['h3_index']))

grid_search = GridSearchCV(
    xgb_base, param_grid,
    cv=cv_splits,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_search.fit(X, y)

# Store best hyperparameters for reuse across all business types
SHARED_BEST_PARAMS = grid_search.best_params_

print(f"\nBest Parameters (from cafe GridSearchCV): {SHARED_BEST_PARAMS}")
print(f"Best ROC-AUC (Spatial CV): {grid_search.best_score_:.3f}")
print("These hyperparameters will be reused for all 6 business types.")


In [ ]:
# ============================================================
# --- Train Per-Type XGBoost Models with Shared Hyperparameters ---
# WHY SHARED HYPERPARAMS: Running GridSearchCV independently for all 6 types would
#   require 6x (grid size) x 5 folds of XGBoost training (~6-8 hours on a laptop).
#   Cafe-tuned hyperparameters generalise well because all types share the same
#   feature space (demographics, centrality, POI co-occurrence) and target structure
#   (sparse binary presence). Only scale_pos_weight is adjusted per type.

# SECTION 7b: Train All 6 Business Types (Shared Hyperparams)
# ============================================================
# Cafe hyperparams were tuned via GridSearchCV above.
# Now train XGBoost for all 6 types using those same hyperparams,
# with type-specific scale_pos_weight for class imbalance.

all_type_results = {}        # {biz_key: {mean_auc, std_auc, y_prob_oof, y_pred_oof, model}}
all_feature_importances = []  # list of dicts for CSV export

for biz_key, biz_cfg in BUSINESS_TYPES.items():
    X_t, y_t, feat_cols = type_datasets[biz_key]
    n_pos = y_t.sum()
    n_neg = len(y_t) - n_pos
    # WHY PER-TYPE IMBALANCE WEIGHT: Each type has different prevalence in London.
    #   Gyms are rarer than cafes (lower positive rate, higher imbalance ratio).
    #   Using cafe ratio for all types would under-penalise FNs for sparse types.

    imbalance_t = n_neg / max(n_pos, 1)

    # Build model with shared hyperparams but type-specific imbalance weight
    model_t = xgb.XGBClassifier(
        **SHARED_BEST_PARAMS,
        scale_pos_weight=imbalance_t,
        eval_metric='logloss',
        random_state=RANDOM_STATE
    )

    # Spatial CV splits (same spatial blocks, different target)
    cv_splits_t = list(spatial_cv.split(X_t, y_t, groups=h3_grid['h3_index']))

    # Out-of-fold predictions for honest evaluation
    # WHY OUT-OF-FOLD PREDICTIONS: If we used model.predict_proba(X_t) on training
    #   data, predictions would be artificially optimistic (model already saw these
    #   examples). OOF predictions use only the held-out fold for each hex, making
    #   evaluation equivalent to a proper test set without wasting data.

    y_prob_oof = cross_val_predict(
        model_t, X_t, y_t, cv=cv_splits_t, method='predict_proba'
    )[:, 1]
    y_pred_oof = (y_prob_oof >= 0.5).astype(int)  # derive from probabilities

    # AUC per fold
    fold_aucs = []
    for train_idx, test_idx in cv_splits_t:
        if y_t.iloc[test_idx].nunique() > 1:
            fold_auc = roc_auc_score(y_t.iloc[test_idx], y_prob_oof[test_idx])
            fold_aucs.append(fold_auc)

    mean_auc = np.mean(fold_aucs) if fold_aucs else 0
    std_auc = np.std(fold_aucs) if fold_aucs else 0

    # Refit on full data for deployment predictions
    # WHY REFIT ON FULL DATA: OOF evaluation uses models that each saw only 80% of
    #   the data. The final deployment model is refitted on ALL data to maximise
    #   training signal. Standard practice: evaluate on OOF, deploy on full-data refit.

    model_t.fit(X_t, y_t)

    # Feature importance
    for feat, imp in zip(feat_cols, model_t.feature_importances_):
        all_feature_importances.append({
            'feature': feat, 'importance': float(imp), 'business_type': biz_key
        })

    all_type_results[biz_key] = {
        'mean_auc': mean_auc,
        'std_auc': std_auc,
        'y_prob_oof': y_prob_oof,
        'y_pred_oof': y_pred_oof,
        'model': model_t,
        'feature_cols': feat_cols,
    }

    print(f"{biz_cfg['label']:25s} "
          f"ROC-AUC = {mean_auc:.3f} +/- {std_auc:.3f}  "
          f"(positive_rate={y_t.mean():.1%}, imbalance={imbalance_t:.1f}:1)")

# Feature importance DataFrame
fi_df = pd.DataFrame(all_feature_importances)
print(f"\nAll {len(BUSINESS_TYPES)} models trained successfully.")
print(f"Feature importances: {len(fi_df)} rows")


# Legacy aliases for downstream cells that reference cafe-specific variables
y_pred_oof = all_type_results['cafe']['y_pred_oof']
y_prob_oof_best = all_type_results['cafe']['y_prob_oof']
tuned_xgb = all_type_results['cafe']['model']
best_model = all_type_results['cafe']['model']


<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 8: Evaluation Suite</h2>
    <p>We present the full evaluation battery: ROC curves, confusion matrix, feature importance, and classification report.</p>

    <table style="width:100%; border-collapse: collapse; margin-top: 10px; font-size: 0.95em;">
        <tr style="background: #1a5276; color: white;">
            <th style="padding: 8px;">Component</th>
            <th style="padding: 8px;">What It Proves</th>
        </tr>
        <tr><td style="padding: 8px;"><strong>ROC Curves</strong></td><td>Discrimination ability across all thresholds (AUC = probability that a random positive is ranked above a random negative)</td></tr>
        <tr style="background: #eaf2f8;"><td style="padding: 8px;"><strong>Confusion Matrices</strong></td><td>Error distribution at the chosen threshold &mdash; reveals whether errors are predominantly FP (over-recommendation) or FN (missed opportunities)</td></tr>
        <tr><td style="padding: 8px;"><strong>Feature Importance</strong></td><td>Which variables drive predictions &mdash; validates that the model uses domain-relevant features (population, centrality) not noise</td></tr>
        <tr style="background: #eaf2f8;"><td style="padding: 8px;"><strong>Calibration</strong></td><td>Whether predicted probabilities match observed frequencies &mdash; essential for threshold tuning to be meaningful</td></tr>
        <tr><td style="padding: 8px;"><strong>Failure Analysis</strong></td><td>Characterises FN/FP feature profiles &mdash; actionable for future feature engineering and model improvement</td></tr>
        <tr style="background: #eaf2f8;"><td style="padding: 8px;"><strong>F-beta Threshold</strong></td><td>Optimises the precision-recall trade-off for the site-selection use case (&beta;=0.3 weights precision 11x over recall)</td></tr>
    </table>
</div>

In [ ]:
# ============================================================
# --- ROC Curves (All Business Types) ---
# WHY OOF PROBABILITIES: Using out-of-fold predictions ensures ROC reflects
#   generalisation performance, not training-set fit. No separate test set exists;
#   spatial CV is the evaluation protocol.

# SECTION 8a: ROC Curves — All Business Types (Spatial CV OOF)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.Set2(np.linspace(0, 1, len(BUSINESS_TYPES)))

for i, (biz_key, result) in enumerate(all_type_results.items()):
    y_t = type_datasets[biz_key][1]
    fpr, tpr, _ = roc_curve(y_t, result['y_prob_oof'])
    auc_val = roc_auc_score(y_t, result['y_prob_oof'])
    label_str = f"{BUSINESS_TYPES[biz_key]['label']} (AUC={auc_val:.3f})"
    ax.plot(fpr, tpr, linewidth=2, color=colors[i], label=label_str)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random Baseline')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: All Business Types (Spatial CV OOF)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/outputs/roc_curves.png', dpi=150, bbox_inches='tight')
fig.savefig('docs/assets/roc_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# ============================================================
# --- Confusion Matrices (Per Business Type) ---
# WHY PER-TYPE MATRICES: Each type has a different base rate and therefore
#   different expected TP/FP/FN/TN counts. A single aggregate matrix would
#   conflate these differences and hide type-specific weaknesses.

# SECTION 8b: Confusion Matrices — All Business Types (Spatial CV OOF)
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (biz_key, result) in zip(axes.ravel(), all_type_results.items()):
    y_t = type_datasets[biz_key][1]
    cm = confusion_matrix(y_t, result['y_pred_oof'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['No', 'Yes'])
    disp.plot(cmap='Blues', ax=ax, values_format='d')
    auc_val = result['mean_auc']
    ax.set_title(f"{BUSINESS_TYPES[biz_key]['label']} (AUC={auc_val:.3f})",
                 fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrices — Spatial CV OOF (All Types)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
fig.savefig('docs/assets/confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Print classification report for primary type (cafe)
y_cafe = type_datasets['cafe'][1]
print("Classification Report — Cafe (Spatial CV OOF):")
print(classification_report(y_cafe, all_type_results['cafe']['y_pred_oof'],
                           target_names=['No Cafe', 'Has Cafe']))


In [ ]:
# ============================================================
# --- Feature Importance (Top 10 per Type) ---
# WHY XGBOOST GAIN IMPORTANCE: Measures average loss improvement per feature
#   across all splits. Fast to compute and interpretable. SHAP would be more
#   rigorous but computationally expensive; gain is the standard for XGBoost.

# SECTION 8c: Feature Importance — All Business Types (Top 10 each)
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, biz_key in zip(axes.ravel(), BUSINESS_TYPES):
    type_fi = fi_df[fi_df['business_type'] == biz_key].nlargest(10, 'importance')
    colors = ['#e94560' if v > type_fi['importance'].median() else '#3498db'
              for v in type_fi['importance'].values]
    ax.barh(type_fi['feature'], type_fi['importance'], color=colors)
    ax.set_title(BUSINESS_TYPES[biz_key]['label'], fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance (Gain)')

plt.suptitle('Top 10 Features by Business Type — Tuned XGBoost', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/outputs/feature_importance.png', dpi=150, bbox_inches='tight')
fig.savefig('docs/assets/feature_importance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("Feature importance rankings saved. Top 5 per type:")
for biz_key in BUSINESS_TYPES:
    top5 = fi_df[fi_df['business_type'] == biz_key].nlargest(5, 'importance')
    features = ', '.join(top5['feature'].tolist())
    print(f"  {biz_key:12s}: {features}")


<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 8d: Calibration &amp; Error Analysis</h2>
    <p>Beyond discrimination (ROC-AUC), we assess <strong>calibration</strong> (are predicted probabilities trustworthy?)
    and <strong>failure modes</strong> (where and why does the model err?).</p>
</div>

In [ ]:
# ============================================================
# --- Calibration Curve (Reliability Diagram) ---
# WHY CHECK CALIBRATION: A model predicting 80% for sites that are actually
#   positive 50% of the time is miscalibrated. The threshold tuning in Section 8f
#   relies on probabilities being approximately calibrated.

# SECTION 8d: Calibration Curve
# ============================================================
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))

# Calibration curve: do predicted probabilities match observed frequencies?
fraction_pos, mean_predicted = calibration_curve(y, y_prob_oof_best, n_bins=8, strategy='uniform')

ax.plot(mean_predicted, fraction_pos, 's-', color='#e94560', linewidth=2, label='Tuned XGBoost')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfectly Calibrated')
ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Observed Fraction of Positives', fontsize=12)
ax.set_title('Calibration Curve — OOF Predictions', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig('docs/assets/calibration_curve.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Interpretation
print("Calibration check:")
for mp, fp_val in zip(mean_predicted, fraction_pos):
    print(f"  Predicted ~{mp:.2f} -> Observed {fp_val:.2f}")
print("\nIf the model says 60% probability, do ~60% of those hexes actually have coffee?")
print("Points above the diagonal = under-confident; below = over-confident.")

In [ ]:
# ============================================================
# Identify False Negatives: existing shops the model failed to predict
# --- Failure Mode Analysis: FP vs TP vs FN Feature Profiles ---
# WHY ANALYSE FALSE NEGATIVES: FNs reveal what kind of location the model
#   systematically misses. If FN sites have lower betweenness than TPs, the model
#   misses cafes in pedestrian backstreets -- an actionable insight for improving
#   the feature set. Bridges evaluation and business domain interpretation.

# SECTION 8e: Failure Mode Analysis — FP vs TP vs FN Feature Profiles
# ============================================================

# Classify OOF predictions by confusion matrix quadrant
oof_outcome = pd.Series('TN', index=y.index)
oof_outcome[(y_pred_oof == 1) & (y == 0)] = 'FP'
oof_outcome[(y_pred_oof == 1) & (y == 1)] = 'TP'
oof_outcome[(y_pred_oof == 0) & (y == 1)] = 'FN'

# Compare feature profiles across outcome groups
analysis_features = ['population', 'level4_perc', 'age_16_to_34_perc',
                     'n_restaurant', 'n_gym', 'nearby_cafe', 'betweenness_centrality']
analysis_features = [f for f in analysis_features if f in X.columns]

outcome_profiles = X.copy()
outcome_profiles['oof_outcome'] = oof_outcome.values

profile_summary = outcome_profiles.groupby('oof_outcome')[analysis_features].mean()

print("Mean Feature Values by OOF Prediction Outcome (Cafe):")
print("=" * 70)
print(profile_summary.round(3).to_string())
print()

# Highlight failure mode insights
if 'FN' in profile_summary.index and 'TP' in profile_summary.index:
    fn_vs_tp = profile_summary.loc['FN'] - profile_summary.loc['TP']
    print("FN vs TP difference (what the model misses):")
    for feat in analysis_features:
        diff = fn_vs_tp[feat]
        direction = "lower" if diff < 0 else "higher"
        print(f"  {feat}: FN has {abs(diff):.3f} {direction} than TP")

print()
if 'FP' in profile_summary.index:
    print(f"False Positive count (OOF): {(oof_outcome == 'FP').sum()}")
    print(f"False Negative count (OOF): {(oof_outcome == 'FN').sum()}")
    print(f"True Positive count (OOF):  {(oof_outcome == 'TP').sum()}")
    print(f"True Negative count (OOF):  {(oof_outcome == 'TN').sum()}")

# Visualise: grouped bar chart of feature means by outcome
fig, ax = plt.subplots(figsize=(12, 6))
profile_norm = profile_summary.div(profile_summary.max())
profile_norm.T.plot(kind='bar', ax=ax, width=0.8,
                    color={'TP': '#e74c3c', 'FP': '#27ae60', 'TN': '#95a5a6', 'FN': '#f39c12'})
ax.set_ylabel('Normalised Mean Value', fontsize=11)
ax.set_title('Feature Profiles by Prediction Outcome (OOF, Cafe)', fontsize=14, fontweight='bold')
ax.legend(title='Outcome', fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
fig.savefig('docs/assets/failure_mode_profiles.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 8f: Precision–Recall Threshold Tuning</h2>
    <p>The default classification threshold of 0.5 is rarely optimal for imbalanced datasets.
    We use the <strong>honest out-of-fold predictions</strong> from spatial cross-validation to find the
    threshold that maximises <strong>F<sub>1</sub></strong> — a metric that balances precision and
    recall equally.</p>
    <p>This balanced approach ensures the model captures most existing businesses
    while still offering high-confidence recommendations, improving map credibility.</p>
</div>

In [ ]:
# ============================================================
# --- Precision-Recall Threshold Tuning (F-beta = 1.0) ---
# WHY BETA=1.0: F-beta = (1+b^2)*P*R / (b^2*P + R). At b=1.0, precision and recall
#   are weighted equally. This balanced threshold ensures the model captures most
#   existing businesses (fewer false negatives / orange dots on the map) while still
#   maintaining reasonable precision. A heavily precision-biased threshold (e.g. b=0.3)
#   would miss too many existing locations, undermining map credibility.

# SECTION 8f: Precision–Recall Threshold Tuning (F1)
# ============================================================
# Use OOF predictions to find thresholds that maximise F-beta (beta=1.0).
# Beta = 1 balances precision and recall for credible coverage on the map.

from sklearn.metrics import precision_recall_curve

optimal_thresholds = {}
beta = 1.0

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (biz_key, result) in zip(axes.ravel(), all_type_results.items()):
    y_t = type_datasets[biz_key][1]
    y_prob_oof = result['y_prob_oof']

    precision, recall, thresholds = precision_recall_curve(y_t, y_prob_oof)

    # F-beta at each threshold (exclude last sentinel element)
    fbeta = ((1 + beta**2) * precision[:-1] * recall[:-1]) / (
        beta**2 * precision[:-1] + recall[:-1] + 1e-8
    )
    best_idx = np.argmax(fbeta)
    optimal_thresholds[biz_key] = float(thresholds[best_idx])

    # Plot Precision-Recall curve
    ax.plot(recall, precision, linewidth=2, color='#2e86c1')
    ax.scatter(recall[best_idx], precision[best_idx],
               color='#e74c3c', s=120, zorder=5, edgecolors='white', linewidth=2)
    ax.annotate(f't={thresholds[best_idx]:.2f}\nF1={fbeta[best_idx]:.3f}',
                xy=(recall[best_idx], precision[best_idx]),
                xytext=(recall[best_idx] - 0.15, precision[best_idx] - 0.10),
                fontsize=9, fontweight='bold', color='#e74c3c',
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5))
    ax.set_xlabel('Recall', fontsize=10)
    ax.set_ylabel('Precision', fontsize=10)
    ax.set_title(f"{BUSINESS_TYPES[biz_key]['label']}\n"
                 f"Optimal t={thresholds[best_idx]:.3f}", fontsize=11, fontweight='bold')
    ax.set_xlim([0, 1.02])
    ax.set_ylim([0, 1.02])
    ax.grid(True, alpha=0.3)

fig.suptitle('Precision\u2013Recall Curves with F1-Optimal Thresholds (Spatial CV OOF)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig('docs/assets/precision_recall_thresholds.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Summary table
print('\nOptimal Thresholds (F1 \u2014 balanced precision-recall):')
print('=' * 65)
for biz_key, t in optimal_thresholds.items():
    label = BUSINESS_TYPES[biz_key]['label']
    print(f'  {label:25s}  threshold = {t:.3f}  (vs default 0.5)')
print(f'\nThese thresholds will be applied in Section 9 deployment predictions.')

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 9: Business Insight — Mining False Positives</h2>
    <p>The most commercially valuable output of this model is not its accuracy — it is its <strong>mistakes</strong>.</p>
    <p>A <strong>False Positive</strong> is a hexagon where:</p>
    <ul>
        <li>The model predicts <code>y=1</code> (this location <em>should</em> have a coffee shop)</li>
        <li>The ground truth is <code>y=0</code> (no coffee shop currently exists here)</li>
    </ul>
    <p>These hexagons possess <em>all the learned features</em> of successful coffee shop locations —
    high footfall, educated demographics, strong transit connectivity, synergy with gyms and offices —
    but <strong>no one has opened a shop there yet</strong>.</p>
    <p>In the language of network theory, these are <strong>Structural Holes</strong> (Burt, 1992) —
    now identified not by heuristic scoring but by supervised machine learning.</p>
</div>

In [ ]:
# ============================================================
# --- Apply Tuned Thresholds & Classify Prediction Outcomes ---
# WHY CONFUSION MATRIX QUADRANTS: The business interpretation differs fundamentally:
#   FP = recommendation (model says suitable, no existing shop = opportunity)
#   TP = existing shop in a good location (validates model pattern)
#   TN = correctly excluded zone (no action needed)
#   FN = model blind spot (shop exists where model did not expect = investigate why)

# SECTION 9: Predictions & Outcomes — All Business Types
# ============================================================
# Apply F1-optimal thresholds from Section 8f instead of default 0.5.
# This balances precision and recall, improving map credibility.

def classify_outcome(pred, actual):
    if pred == 1 and actual == 0: return 'False Positive (Recommendation)'
    if pred == 1 and actual == 1: return 'True Positive'
    if pred == 0 and actual == 0: return 'True Negative'
    return 'False Negative'

for biz_key in BUSINESS_TYPES:
    X_t, y_t, _ = type_datasets[biz_key]
    model = all_type_results[biz_key]['model']
    threshold = optimal_thresholds[biz_key]

    y_prob = model.predict_proba(X_t)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    # Compare against default 0.5 for diagnostics
    y_pred_default = (y_prob >= 0.5).astype(int)
    n_fp_default = int(((y_pred_default == 1) & (y_t == 0)).sum())
    n_fp_tuned = int(((y_pred == 1) & (y_t == 0)).sum())

    h3_grid[f'predicted_prob_{biz_key}'] = y_prob
    h3_grid[f'predicted_{biz_key}'] = y_pred

    h3_grid[f'outcome_{biz_key}'] = [
        classify_outcome(int(p), int(a))
        for p, a in zip(y_pred, y_t.values)
    ]

    n_fp = (h3_grid[f'outcome_{biz_key}'] == 'False Positive (Recommendation)').sum()
    n_tp = (h3_grid[f'outcome_{biz_key}'] == 'True Positive').sum()
    print(f"{BUSINESS_TYPES[biz_key]['label']:25s} "
          f"FP={n_fp:4d} (was {n_fp_default} at t=0.5), TP={n_tp:4d}, "
          f"threshold={threshold:.3f}")

# Legacy aliases (point to cafe for backward compat)
h3_grid['outcome'] = h3_grid['outcome_cafe']
h3_grid['predicted_prob'] = h3_grid['predicted_prob_cafe']
h3_grid['predicted'] = h3_grid['predicted_cafe']
h3_grid['actual'] = h3_grid['has_cafe']

# Print cafe outcome distribution (primary type)
print(f"\nCafe Outcome Distribution (threshold={optimal_thresholds['cafe']:.3f}):")
print(h3_grid['outcome_cafe'].value_counts().to_string())

# Extract and rank False Positives for primary type
fp = h3_grid[h3_grid['outcome_cafe'] == 'False Positive (Recommendation)'].copy()
fp_ranked = fp.nlargest(10, 'predicted_prob_cafe')

print(f"\nTOP 10 CAFE RECOMMENDATIONS (False Positives):")
display_cols = ['h3_index', 'borough', 'predicted_prob_cafe', 'population',
                'level4_perc', 'n_restaurant', 'n_gym', 'nearby_cafe']
print(fp_ranked[[c for c in display_cols if c in fp_ranked.columns]].to_string(index=False))

In [ ]:
# ============================================================
# --- Demographic Profile of Recommended Sites ---
# WHY COMPARE FP vs TP vs LONDON AVERAGE: If recommended (FP) sites are
#   demographically similar to proven (TP) sites, this validates that the model
#   recommends locations with genuine demand-side similarity to existing locations,
#   not random hexes. Divergence would require investigation.

# SECTION 9b: Demographic Profile of Recommended Sites (Cafe)
# ============================================================
# Compare FP sites vs London average (primary type: cafe)
profile_cols = ['population', 'level4_perc', 'age_16_to_34_perc',
                'employed_total_perc', 'betweenness_centrality',
                'n_restaurant', 'n_gym', 'n_station', 'nearby_cafe']

available_cols = [c for c in profile_cols if c in h3_grid.columns]

comparison = pd.DataFrame({
    'London Average': h3_grid[available_cols].mean(),
    'Recommended Sites (FP)': fp[available_cols].mean() if len(fp) > 0 else 0,
    'Existing Cafes (TP)': h3_grid[h3_grid['outcome_cafe'] == 'True Positive'][available_cols].mean()
})

print("Demographic Profile Comparison (Cafe):")
comparison


<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Task 2: Detecting Structural Holes (Heuristic Site Score)</h2>
    <p>A <strong>Structural Hole</strong> (Burt, 1992) is a location with high demand signals but no existing supply.
    We quantify this with a heuristic score per hexagon:</p>
    <p style="text-align: center; font-size: 1.1em;">
        $$Score_H = Pop_H \times D_H + \alpha \cdot S_H + \beta \cdot A_H - \gamma \cdot C_H$$
    </p>
    <table style="width:100%; border-collapse: collapse; margin: 15px 0; font-size: 0.95em;">
        <tr style="background: #884ea0; color: white;">
            <th style="padding: 8px;">Symbol</th><th style="padding: 8px;">Meaning</th><th style="padding: 8px;">Value</th>
        </tr>
        <tr><td style="padding: 8px;">$Pop_H$</td><td>LandScan population count</td><td>From zonal stats</td></tr>
        <tr style="background: #f4ecf7;"><td style="padding: 8px;">$D_H$</td><td>Demand proxy: <code>level4_perc / 100</code></td><td>[0, 1]</td></tr>
        <tr><td style="padding: 8px;">$\alpha$</td><td>Synergy weight (gyms, universities, offices)</td><td>5</td></tr>
        <tr style="background: #f4ecf7;"><td style="padding: 8px;">$\beta$</td><td>Anchor weight (transit stations)</td><td>3</td></tr>
        <tr><td style="padding: 8px;">$\gamma$</td><td>Competitor penalty (cafes, coffee shops)</td><td>15</td></tr>
    </table>
    <p><strong>Note:</strong> This is an <em>interpretable heuristic</em>. The ML notebook
    (<code>camden_predictive_model.ipynb</code>) learns optimal weights from the data using supervised classification.
    POIs are classified using the <code>role</code> column created in Notebook 01.</p>
</div>

In [ ]:
# ============================================================
# --- Heuristic Site Score (Transparent Alternative to ML) ---
# WHY HEURISTIC ALONGSIDE ML MODEL: The XGBoost model is a black box to non-technical
#   stakeholders. This heuristic provides a transparent, auditable linear formula that
#   produces similar rankings. Useful for "explain why this location" conversations.

# Heuristic Site Score (Structural Hole Detection)
# ============================================================
# Independent of the ML model — a transparent, auditable formula.
# Demand proxy: degree-level education normalised to [0,1]
h3_grid['demand_index'] = h3_grid['level4_perc'] / 100.0

ALPHA, BETA, GAMMA = 5, 3, 15
h3_grid['site_score'] = (
    h3_grid['population'] * h3_grid['demand_index']
    + ALPHA * h3_grid['n_synergy']
    + BETA  * h3_grid['n_anchors']
    - GAMMA * h3_grid['n_competitors']
)

print(f"Heuristic Site Score range: [{h3_grid['site_score'].min():.1f}, {h3_grid['site_score'].max():.1f}]")
print(f"Mean: {h3_grid['site_score'].mean():.1f}")
print(f"Positive-score hexes: {(h3_grid['site_score'] > 0).sum()} / {len(h3_grid)}")

<div style="margin-top: 30px;">
    <h2 style="color: #e94560; border-bottom: 2px solid #e94560; padding-bottom: 10px;">Section 10: Interactive 3D Recommendation Map</h2>
    <p>This map translates the model's predictions into a <strong>spatial decision tool</strong>. Each hexagon across Greater London is classified by comparing the model's prediction against ground truth:</p>

    <table style="width:100%; border-collapse: collapse; margin: 15px 0; font-size: 0.95em;">
        <tr style="background: #16213e; color: white;">
            <th style="padding: 8px;">Colour</th>
            <th style="padding: 8px;">Category</th>
            <th style="padding: 8px;">What It Means</th>
            <th style="padding: 8px;">Business Action</th>
        </tr>
        <tr style="background: #d5f5e3;">
            <td style="padding: 8px; text-align: center;"><strong style="color: #27ae60; font-size: 1.3em;">&#9632;</strong> Green (tall)</td>
            <td><strong>False Positive</strong><br><em>Model says YES, reality says NO</em></td>
            <td>This hex has all the features of a successful coffee location &mdash; but no shop exists here yet.</td>
            <td style="color: #27ae60; font-weight: bold;">Investigate as a new site opportunity</td>
        </tr>
        <tr>
            <td style="padding: 8px; text-align: center;"><strong style="color: #e74c3c; font-size: 1.3em;">&#9632;</strong> Red (flat)</td>
            <td><strong>True Positive</strong><br><em>Model says YES, reality says YES</em></td>
            <td>The model correctly identified an existing coffee shop location. Validates the model.</td>
            <td>No action &mdash; already served</td>
        </tr>
        <tr style="background: #f9f9f9;">
            <td style="padding: 8px; text-align: center;"><strong style="color: #f39c12; font-size: 1.3em;">&#9632;</strong> Orange (flat)</td>
            <td><strong>False Negative</strong><br><em>Model says NO, reality says YES</em></td>
            <td>A coffee shop exists here but the model didn't predict it. The model underestimates this area.</td>
            <td>Flag as model blind spot</td>
        </tr>
        <tr>
            <td style="padding: 8px; text-align: center;"><strong style="color: #95a5a6; font-size: 1.3em;">&#9632;</strong> Grey (flat)</td>
            <td><strong>True Negative</strong><br><em>Model says NO, reality says NO</em></td>
            <td>No coffee shop, and the model agrees this location lacks the demand-side signals.</td>
            <td>Low priority</td>
        </tr>
    </table>

    <p><strong>Height encoding:</strong> Green (FP) hexagons are <em>extruded upward</em> proportional to the model's confidence score.
    Taller columns = higher predicted probability = stronger recommendation. All other categories are rendered flat.</p>

    <p><strong>How to read the map:</strong> Look for <strong>tall green clusters</strong> &mdash; these are the neighbourhoods where the model sees
    the strongest untapped opportunity. Hover over any hexagon for detailed statistics.</p>

    <div style="background: #f5eef8; padding: 12px; border-radius: 5px; border-left: 4px solid #7d3c98; margin-top: 10px;">
        <strong>Why Pydeck over Kepler.gl or Folium?</strong>
        <ul style="margin: 5px 0;">
            <li><strong>Pydeck</strong>: Generates self-contained HTML files with WebGL rendering. Handles 16,000+ hexagons with smooth 60fps performance. Supports 3D extrusion (height = confidence) natively via <code>H3HexagonLayer</code>.</li>
            <li><strong>Kepler.gl</strong>: Requires a running Jupyter kernel for interactivity; exported HTML files are static and lose filtering/layer controls. Not suitable for portfolio embedding.</li>
            <li><strong>Folium</strong>: Leaflet-based (2D only). No native H3 hexagon support or 3D extrusion. Rendering 16k polygons causes significant browser lag.</li>
        </ul>
    </div>
</div>

In [ ]:
# ============================================================
# --- Interactive 3D Recommendation Maps (All Business Types) ---
# WHY CONFUSION MATRIX COLOURING: Each hexagon is coloured by its prediction outcome:
#   Green (FP) = recommendation (model predicts suitable, no existing shop)
#   Red (TP) = existing business correctly identified by model
#   Grey (TN) = correctly excluded zone (no shop, model agrees)
#   Orange (FN) = model blind spot (shop exists but model missed it)

# SECTION 10: Pydeck 3D Recommendation Maps
# ============================================================
# Generates 6 interactive HTML maps with:
#   - Borough filter dropdown (in-map, no reload needed)
#   - Confidence threshold slider (live filter on FP recommendations)
#   - Competitive opportunity score for TP hexagons
#   - Existing business count in tooltip

import os
os.makedirs('data/outputs', exist_ok=True)

# Sorted list of all 33 London boroughs for the filter dropdown
LONDON_BOROUGHS = [
    'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley',
    'Camden', 'City of London', 'City of Westminster', 'Croydon', 'Ealing',
    'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey',
    'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington',
    'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham',
    'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames', 'Southwark',
    'Sutton', 'Tower Hamlets', 'Waltham Forest', 'Wandsworth',
]

for biz_key, biz_info in BUSINESS_TYPES.items():
    biz_label = biz_info['label']
    outcome_col = f'outcome_{biz_key}'
    prob_col = f'predicted_prob_{biz_key}'

    viz_df = h3_grid.to_crs(epsg=4326).copy()

    # Map type-specific columns to generic names
    viz_df['outcome'] = viz_df[outcome_col]
    viz_df['predicted_prob'] = viz_df[prob_col]

    # Feature: existing business counts for tooltip (n_similar = in-hex, nearby_similar = 350m ring)
    n_col = f'n_{biz_key}'
    nearby_col = f'nearby_{biz_key}'
    viz_df['n_similar'] = viz_df[n_col].fillna(0).astype(int) if n_col in viz_df.columns else 0
    viz_df['nearby_similar'] = viz_df[nearby_col].fillna(0).astype(int) if nearby_col in viz_df.columns else 0

    # Competitive opportunity score for TP hexagons:
    # score = predicted_prob / (1 + nearby_count). Higher = proven demand, lower saturation.
    tp_mask = viz_df['outcome'] == 'True Positive'
    viz_df['comp_score'] = 0.0
    if tp_mask.sum() > 0:
        viz_df.loc[tp_mask, 'comp_score'] = (
            viz_df.loc[tp_mask, 'predicted_prob'] /
            (1 + viz_df.loc[tp_mask, 'nearby_similar'])
        ).round(3)
    viz_df['comp_label'] = 'N/A'
    if tp_mask.sum() > 0:
        viz_df.loc[tp_mask, 'comp_label'] = viz_df.loc[tp_mask, 'comp_score'].astype(str)

    # Tooltip: human-readable outcome labels
    co_loc_label = 'Existing ' + biz_label.lower() + ' \u2014 co-location opportunity'
    outcome_labels = {
        'False Positive (Recommendation)': 'RECOMMENDED SITE (no shop yet)',
        'True Positive':                   co_loc_label,
        'True Negative':                   'Not suitable (model agrees)',
        'False Negative':                  'Model blind spot (shop exists, model missed)',
    }
    viz_df['outcome_label'] = viz_df['outcome'].map(outcome_labels)

    # Colour coding by confusion matrix quadrant
    color_map = {
        'False Positive (Recommendation)': [39, 174, 96, 200],
        'True Positive':                   [231, 76, 60, 160],
        'True Negative':                   [149, 165, 166, 80],
        'False Negative':                  [243, 156, 18, 160],
    }
    viz_df['color'] = viz_df['outcome'].map(color_map)

    # TP gradient: bright red = high comp_score (best co-location), faded = lower opportunity
    if tp_mask.sum() > 0:
        tp_median = viz_df.loc[tp_mask, 'comp_score'].median()
        high_tp = tp_mask & (viz_df['comp_score'] >= tp_median)
        low_tp  = tp_mask & (viz_df['comp_score'] <  tp_median)
        viz_df.loc[high_tp, 'color'] = pd.Series(
            [[231, 76, 60, 220]] * int(high_tp.sum()), index=viz_df.index[high_tp]
        )
        viz_df.loc[low_tp, 'color'] = pd.Series(
            [[231, 76, 60, 80]] * int(low_tp.sum()), index=viz_df.index[low_tp]
        )

    # FP confidence floor: dim borderline recommendations
    fp_mask = viz_df['outcome'] == 'False Positive (Recommendation)'
    if fp_mask.sum() > 0:
        fp_median_prob = viz_df.loc[fp_mask, 'predicted_prob'].median()
        low_conf_fp = fp_mask & (viz_df['predicted_prob'] < fp_median_prob)
        viz_df.loc[low_conf_fp, 'color'] = pd.Series(
            [[39, 174, 96, 60]] * int(low_conf_fp.sum()),
            index=viz_df.index[low_conf_fp]
        )
        viz_df.loc[low_conf_fp, 'outcome_label'] = 'Borderline recommendation (lower confidence)'

    # Elevation: FPs extruded by confidence, all others flat
    viz_df['elevation'] = viz_df.apply(
        lambda r: r['predicted_prob'] * 500
        if r['outcome'] == 'False Positive (Recommendation)' else 10,
        axis=1
    )
    if fp_mask.sum() > 0:
        viz_df.loc[low_conf_fp, 'elevation'] = 10

    # Round columns for cleaner tooltips
    viz_df['prob_pct'] = (viz_df['predicted_prob'] * 100).round(1)
    viz_df['pop_display'] = viz_df['population'].round(0).astype(int)
    viz_df['level4_display'] = viz_df['level4_perc'].round(1)
    viz_df['age_display'] = viz_df['age_16_to_34_perc'].round(1)

    # Only render non-TN hexagons (TNs are ~80% and add browser load with no insight)
    viz_render = viz_df[viz_df['outcome'] != 'True Negative'].copy()

    layer = pdk.Layer(
        'H3HexagonLayer',
        viz_render,
        pickable=True,
        stroked=True,
        filled=True,
        extruded=True,
        get_hexagon='h3_index',
        get_fill_color='color',
        get_elevation='elevation',
        elevation_scale=1,
    )

    view_state = pdk.ViewState(
        latitude=51.509, longitude=-0.118,
        zoom=10.0, pitch=50, bearing=-15
    )

    # Tooltip HTML - note: {col} placeholders are for Pydeck (not Python f-strings)
    existing_line = '<b>Existing ' + biz_label.lower() + 's in hex:</b> {n_similar}<br>'
    tooltip = {
        'html': (
            '<div style="font-family: Arial, sans-serif;">'
            '<div style="font-size: 14px; font-weight: bold; margin-bottom: 6px; '
            'padding-bottom: 4px; border-bottom: 1px solid rgba(255,255,255,0.3);">'
            '{outcome_label}</div>'
            '<div style="font-size: 12px; line-height: 1.6;">'
            '<b>Borough:</b> {borough}<br>'
            '<b>Confidence:</b> {prob_pct}%<br>'
            + existing_line +
            '<b>Nearby (350m):</b> {nearby_similar}<br>'
            '<b>Co-location score:</b> {comp_label}<br>'
            '<b>Population:</b> {pop_display}<br>'
            '<b>Degree-educated:</b> {level4_display}%<br>'
            '<b>Age 16-34:</b> {age_display}%<br>'
            '<b>Synergy POIs:</b> {n_synergy} &nbsp;|&nbsp; <b>Anchors:</b> {n_anchors}'
            '</div>'
            '</div>'
        ),
        'style': {
            'backgroundColor': '#1a1a2e',
            'color': 'white',
            'fontSize': '12px',
            'padding': '10px',
            'borderRadius': '6px'
        }
    }

    # Build legend + controls HTML injected into the map
    legend_title = 'London ' + biz_label + ' Site Selection'
    existing_label = 'Existing ' + biz_label.lower() + 's'
    bor_options = ''.join(
        '<option value="' + b + '">' + b + '</option>' for b in LONDON_BOROUGHS
    )

    CONTROLS_HTML = (
        '<div id="map-controls" style="position: fixed; bottom: 220px; left: 20px; z-index: 1001;'
        ' background: rgba(26,26,46,0.95); padding: 12px 14px; border-radius: 8px;'
        ' font-family: Arial, sans-serif; font-size: 12px; color: white;'
        ' border: 1px solid rgba(233,69,96,0.5); min-width: 240px;">'
        '<div style="font-weight: bold; font-size: 12px; margin-bottom: 8px;'
        ' border-bottom: 1px solid rgba(255,255,255,0.15); padding-bottom: 5px;">'
        ' Filter Controls</div>'
        '<div style="margin-bottom: 8px;">'
        '<label style="display:block; margin-bottom:3px; color:#ccc;">Borough</label>'
        '<select id="borSel" style="width:100%; padding:4px; background:#0f3460;'
        ' color:white; border:1px solid #e94560; border-radius:4px; font-size:11px;'
        ' cursor:pointer;" onchange="window._rebuildDeck(this.value,'
        ' parseFloat(document.getElementById(\"thrSl\").value))">'
        '<option value="All">All London</option>'
        + bor_options +
        '</select>'
        '</div>'
        '<div>'
        '<label style="display:block; margin-bottom:3px; color:#ccc;">'
        'Min recommendation confidence: <span id="thrLbl" style="color:#e94560;'
        ' font-weight:bold;">0%</span></label>'
        '<input type="range" id="thrSl" min="0" max="0.99" step="0.05" value="0"'
        ' style="width:100%; accent-color:#e94560; cursor:pointer;"'
        ' oninput="document.getElementById(\"thrLbl\").textContent='
        'Math.round(this.value*100)+\'%\';'
        ' window._rebuildDeck(document.getElementById(\"borSel\").value,'
        ' parseFloat(this.value));">'
        '</div>'
        '</div>'
    )

    LEGEND_HTML = (
        '<div style="position: fixed; bottom: 20px; left: 20px; z-index: 1000;'
        ' background: rgba(26,26,46,0.92); padding: 14px 18px; border-radius: 8px;'
        ' font-family: Arial, sans-serif; font-size: 12px; color: white;'
        ' border: 1px solid rgba(233,69,96,0.4); max-width: 260px;">'
        '<div style="font-weight: bold; font-size: 13px; margin-bottom: 8px;'
        ' border-bottom: 1px solid rgba(255,255,255,0.2); padding-bottom: 6px;">'
        + legend_title +
        '</div>'
        '<div style="margin-bottom: 4px;">'
        '<span style="color: #27ae60; font-size: 16px;">&#9632;</span> '
        '<b>Green (tall)</b> \u2014 High-confidence recommendations'
        '</div>'
        '<div style="margin-bottom: 4px;">'
        '<span style="color: rgba(39,174,96,0.82); font-size: 16px;">&#9632;</span> '
        '<b>Faint green</b> \u2014 Borderline recommendations'
        '</div>'
        '<div style="margin-bottom: 4px;">'
        '<span style="color: #e74c3c; font-size: 16px;">&#9632;</span> '
        '<b>Bright red</b> \u2014 Best co-location spots (high demand)'
        '</div>'
        '<div style="margin-bottom: 4px;">'
        '<span style="color: rgba(231,76,60,0.72); font-size: 16px;">&#9632;</span> '
        '<b>Faded red</b> \u2014 ' + existing_label + ' (lower opportunity)'
        '</div>'
        '<div style="margin-bottom: 4px;">'
        '<span style="color: #f39c12; font-size: 16px;">&#9632;</span> '
        '<b>Orange</b> \u2014 Model blind spots'
        '</div>'
        '<div style="font-size: 11px; color: #ccc; border-top: 1px solid rgba(255,255,255,0.15);'
        ' padding-top: 6px;">'
        'Height = model confidence | Taller = stronger recommendation<br>'
        'Threshold: F<sub>1</sub>-optimal | Hover for full stats'
        '</div>'
        '</div>'
    )

    deck = pdk.Deck(
        layers=[layer],
        initial_view_state=view_state,
        tooltip=tooltip,
    )

    html_path = f'data/outputs/london_recommendations_{biz_key}.html'
    deck.to_html(html_path)

    # Post-process: inject controls + legend + borough/threshold filter JS
    with open(html_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    # Inject rebuild JS before deck instantiation so _fullData and _rebuildDeck are global
    # WHY: The borough and threshold filter work by re-creating the deck with filtered data.
    #   We store the full dataset once, then filter on each user interaction.
    REBUILD_JS = (
        'window._fullData = JSON.parse(JSON.stringify(jsonInput.layers[0].data));\n'
        'window._rebuildDeck = function(bor, thr) {\n'
        '  var fd = window._fullData.filter(function(d) {\n'
        '    var borOk = bor === "All" || d.borough === bor;\n'
        '    var isFP = d.outcome === "False Positive (Recommendation)";\n'
        '    var thrOk = !isFP || (d.predicted_prob >= thr);\n'
        '    return borOk && thrOk;\n'
        '  });\n'
        '  jsonInput.layers[0].data = fd;\n'
        '  if (window._deckInst) { try { window._deckInst.finalize(); } catch(e) {} }\n'
        '  container.innerHTML = '';\n'
        '  window._deckInst = createDeck({container: container, jsonInput: jsonInput,\n'
        '    tooltip: tooltip, customLibraries: null, configuration: null});\n'
'};\n'
        'window.addEventListener("message", function(ev) {\n'
        '  if (ev.data && ev.data.type === "rebuildDeck") {\n'
        '    _rebuildDeck(ev.data.borough, ev.data.threshold);\n'
        '  }\n'
        '});\n'
    )

    html_content = html_content.replace(
        'const deckInstance = createDeck(',
        REBUILD_JS + 'window._deckInst = createDeck('
    )

    # Inject controls panel and legend before </body>
    html_content = html_content.replace(
        '</body>',
        LEGEND_HTML + '</body>'
    )

    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    # Print summary
    outcome_counts = viz_df['outcome'].value_counts()
    n_fp = outcome_counts.get('False Positive (Recommendation)', 0)
    n_fn = outcome_counts.get('False Negative', 0)
    n_tp = outcome_counts.get('True Positive', 0)
    print(f'{biz_label:25s} | FP={n_fp:4d} TP={n_tp:4d} FN={n_fn:4d} | '
          f'Rendered={len(viz_render):5d} | {html_path}')

print()
print(f'All {len(BUSINESS_TYPES)} maps saved to data/outputs/')
print('Borough filter + confidence slider injected into each map')
print('Hover red hexagons to see co-location opportunity scores')


<div style="margin-top: 30px; padding: 20px; background-color: #1a1a2e; border-radius: 12px; border: 2px solid #e94560;">
    <h2 style="color: #ff6b82;">Section 11: Model Card</h2>
    <p style="color: #e0e0e0;">Following Mitchell et al. (2019), <em>Model Cards for Model Reporting</em>.</p>

    <div style="background: #16213e; padding: 15px; border-radius: 8px; margin-top: 10px; color: #f5f5f5;">
        <h3 style="color: #ff6b82;">1. Model Purpose</h3>
        <p>Binary classifier identifying underserved H3 hexagonal cells for specialty coffee retail
        across all 33 London boroughs (Greater London). Intended users: B2B site selection analysts, commercial real
        estate teams, and urban planning consultants.</p>
        <p><strong>Use case:</strong> The model's False Positive predictions (FP) represent <em>structural holes</em>
        &mdash; locations that possess the demand-side characteristics of existing coffee shop sites but
        where no shop currently operates. These are candidate sites for new investment.</p>

        <h3 style="color: #ff6b82; margin-top: 15px;">2. Data Provenance</h3>
        <table style="width:100%; color: #e0e0e0; border-collapse: collapse; font-size: 0.9em;">
            <tr style="border-bottom: 1px solid #333;">
                <td style="padding: 6px;"><strong>Population</strong></td>
                <td>LandScan Global 2022 (Oak Ridge National Laboratory), ~1 km resolution GeoTIFF</td>
            </tr>
            <tr style="border-bottom: 1px solid #333;">
                <td style="padding: 6px;"><strong>Demographics</strong></td>
                <td>ONS Census 2021 (EDINA Digimap): age structure, economic activity, qualifications. Output Area level (~125 m median diameter)</td>
            </tr>
            <tr style="border-bottom: 1px solid #333;">
                <td style="padding: 6px;"><strong>Points of Interest</strong></td>
                <td>OpenStreetMap via OSMnx (snapshot at notebook runtime). Tags: amenity, leisure, shop, public_transport</td>
            </tr>
            <tr style="border-bottom: 1px solid #333;">
                <td style="padding: 6px;"><strong>Spatial Unit</strong></td>
                <td>Uber H3 Resolution 9 hexagons (~174 m edge length, ~0.105 km&sup2;)</td>
            </tr>
            <tr>
                <td style="padding: 6px;"><strong>Graph Features</strong></td>
                <td>H3 adjacency graph via NetworkX. 6 metrics: degree, betweenness, closeness centrality, clustering coefficient, <strong>eigenvector centrality</strong> ($\mathbf{A}\mathbf{x}=\lambda_1\mathbf{x}$), and <strong>PageRank</strong> (damped random-walk probability, $\alpha=0.85$)</td>
            </tr>
        </table>

        <h3 style="color: #ff6b82; margin-top: 15px;">3. Evaluation Summary</h3>
        <ul>
            <li><strong>Best model:</strong> XGBoost (tuned via GridSearchCV)</li>
            <li><strong>Feature count:</strong> 14 features across 4 modalities</li>
            <li><strong>Evaluation protocol:</strong> 5-fold Spatial Block CV using H3 Res-7 parent partitioning with greedy class-balanced fold assignment</li>
            <li><strong>Primary metric:</strong> ROC-AUC (out-of-fold predictions)</li>
            <li><strong>Note:</strong> Spatial CV typically reduces apparent AUC by 5&ndash;15% compared to random CV due to controlling for spatial autocorrelation. This is a feature, not a bug &mdash; it reflects honest generalisation performance.</li>
        </ul>

        <h3 style="color: #ff6b82; margin-top: 15px;">4. Known Limitations</h3>
        <ul style="color: #f0f0f0;">
            <li><strong>Geographic scope:</strong> Trained on all 33 London boroughs (Greater London). The model may not generalise to other UK cities without retraining, as the feature&ndash;target relationship is conditioned on London's urban structure.</li>
            <li><strong>Temporal snapshot:</strong> Census data is from 2021; OSM data from the notebook run date. The model does not capture neighbourhood evolution, new developments, or post-pandemic shifts in foot traffic.</li>
            <li><strong>Binary target:</strong> Coffee shop presence &ne; profitability. A hexagon with a struggling caf&eacute; counts as positive (y=1). No revenue, footfall volume, or commercial rent data is incorporated.</li>
            <li><strong>OSM completeness:</strong> OpenStreetMap coverage varies. Some POIs (especially offices and gyms) may be under-reported, affecting <code>n_synergy</code> accuracy.</li>
            <li><strong>Omitted variables:</strong> Commercial rent, planning restrictions, lease availability, competitor brand strength, and pavement footfall counts are not modelled. The model identifies <em>demand-side</em> opportunity only.</li>
            <li><strong>No held-out test set:</strong> All data is used within spatial CV. There is no temporally or geographically independent test set to validate deployment performance.</li>
        </ul>

        <h3 style="color: #ff6b82; margin-top: 15px;">5. Ethical Considerations</h3>
        <ul style="color: #f0f0f0;">
            <li><strong>Gentrification risk:</strong> Recommending specialty coffee in underserved areas could contribute to gentrification pressure, potentially displacing existing businesses or communities. Decision-makers must consider local impact.</li>
            <li><strong>Algorithmic bias:</strong> The model may encode demographic proxies. Areas with lower educational attainment are systematically ranked lower &mdash; this reflects market demand patterns but should not be interpreted as a normative judgement about community value.</li>
            <li><strong>Human-in-the-loop:</strong> Predictions should <strong>inform</strong>, not replace, human decision-making. Field visits, stakeholder consultation, and financial due diligence must precede any investment decision.</li>
        </ul>

        <h3 style="color: #ff6b82; margin-top: 15px;">6. Recommended Use</h3>
        <p style="color: #f0f0f0;">This model is a <strong>shortlisting tool</strong>. It narrows the search space for site analysts
        by identifying the most promising hexagonal zones. It must be combined with:
        (a) field visits, (b) commercial rent analysis, (c) local authority planning checks, and
        (d) community stakeholder consultation before any investment commitment.</p>
    </div>
</div>

In [ ]:
# ============================================================
# --- Export Final Results for Downstream Consumers ---
# Outputs: Parquet grid, per-type FP CSVs, feature importance, model comparison table.
# These files are consumed by the Streamlit dashboard and portfolio site.

# SECTION 12: Export Final Results (Multi-Type)
# ============================================================

# 1. Full grid with all per-type predictions
# WHY PARQUET NOT CSV: The GeoDataFrame contains geometry columns (Shapely Polygon)
#   that cannot be serialised in CSV. Parquet preserves dtypes including geometry,
#   enabling fast read-back without re-running the pipeline. ~40-60% smaller than CSV.

# WHY PARQUET: Preserves geometry columns, 10x compression vs CSV, fast read-back
h3_grid.to_parquet('data/outputs/london_ml_scored.parquet')

# 2. Feature importances CSV (per-type)
# CSV export: human-readable format for non-technical stakeholders (Excel-compatible)
fi_df.to_csv('data/outputs/feature_importances.csv', index=False)

# 3. Extended model comparison (all types + cafe baselines)
model_comp_rows = []
for biz_key, result in all_type_results.items():
    model_comp_rows.append({
        'Model': 'XGBoost (Tuned)',
        'business_type': biz_key,
        'Mean AUC': result['mean_auc'],
        'Std AUC': result['std_auc'],
    })
# Also include cafe LR and RF from Cell 30
if 'results' in dir():
    for name, res in results.items():
        model_comp_rows.append({
            'Model': name,
            'business_type': 'cafe',
            'Mean AUC': res['mean_auc'],
            'Std AUC': res['std_auc'],
        })
model_comp_df = pd.DataFrame(model_comp_rows)
model_comp_df.to_csv('data/outputs/model_comparison.csv', index=False)

# Top 50 per type: practical shortlist for a site analyst to investigate on foot
# 4. Per-type FP recommendations (top 50 each)
for biz_key in BUSINESS_TYPES:
    outcome_col = f'outcome_{biz_key}'
    prob_col = f'predicted_prob_{biz_key}'
    fp_mask = h3_grid[outcome_col] == 'False Positive (Recommendation)'
    fp_type = h3_grid[fp_mask].copy()
    if len(fp_type) > 0:
        # WHY TOP 50 PER TYPE: 50 sites is a practical shortlist for a B2B site analyst --
        #   large enough for geographic diversity, small enough to review in one session.
        #   nlargest on predicted_prob ensures highest-confidence recommendations included.

        fp_top = fp_type.nlargest(min(50, len(fp_type)), prob_col)
        fp_export = fp_top.to_crs(epsg=4326)
        fp_export['latitude'] = fp_export.geometry.centroid.y
        fp_export['longitude'] = fp_export.geometry.centroid.x
        fp_export.to_csv(f'data/outputs/fp_recommendations_{biz_key}.csv', index=False)
        print(f"  fp_recommendations_{biz_key}.csv: {len(fp_top)} sites")

# 5. Legacy cafe-only export (backward compat)
fp_cafe = h3_grid[h3_grid['outcome_cafe'] == 'False Positive (Recommendation)']
if len(fp_cafe) > 0:
    fp_top = fp_cafe.nlargest(50, 'predicted_prob_cafe').copy()
    fp_export = fp_top.to_crs(epsg=4326)
    fp_export['latitude'] = fp_export.geometry.centroid.y
    fp_export['longitude'] = fp_export.geometry.centroid.x
    export_cols = [
        'h3_index', 'borough', 'latitude', 'longitude', 'predicted_prob_cafe',
        'population', 'level4_perc', 'age_16_to_34_perc',
        'employed_total_perc', 'n_restaurant', 'n_gym', 'n_station',
        'betweenness_centrality', 'closeness_centrality',
        'eigenvector_centrality', 'pagerank', 'nearby_cafe'
    ]
    fp_export[[c for c in export_cols if c in fp_export.columns]].to_csv(
        'data/outputs/fp_recommendations.csv', index=False
    )

# Confirmation message: log the output path for reproducibility audit trail
print(f"\nFinal outputs saved:")
print(f"  london_ml_scored.parquet      ({len(h3_grid)} hexes, {h3_grid.shape[1]} columns, "
      f"{h3_grid['borough'].nunique()} boroughs)")
print(f"  feature_importances.csv       ({len(fi_df)} rows)")
print(f"  model_comparison.csv          ({len(model_comp_df)} rows)")
print(f"  london_ml_recommendations.html (interactive 3D map)")
for biz_key in BUSINESS_TYPES:
    n_fp = (h3_grid[f'outcome_{biz_key}'] == 'False Positive (Recommendation)').sum()
    print(f"  {biz_key:12s}: {n_fp} FP recommendations")
print(f"\nPipeline complete. {len(BUSINESS_TYPES)} business types modelled.")


In [ ]:
# ============================================================
# --- Export Model Report Data for Portfolio 'Model Report' Tab ---
# WHY SEPARATE JSON: The HTML portfolio page needs thresholds, outcome counts,
#   and top recommendations in a machine-readable format. These are not in the
#   CSVs already exported. Writing a single JSON avoids re-running the full
#   pipeline just to update the portfolio page.

# SECTION 13: Export docs/model_report_data.json
# ============================================================
import json as _json, datetime

report = {
    'generated_at': datetime.datetime.now().isoformat(),
    'grid': {
        'hexagons': int(len(h3_grid)),
        'boroughs': int(h3_grid['borough'].nunique()),
        'resolution': 9,
        'crs': 'EPSG:27700',
        'features': 23,
    },
    'thresholds': {k: round(v, 4) for k, v in optimal_thresholds.items()},
    'auc_scores': {},
    'outcomes': {},
    'top_features': {},
    'top_recommendations': {},
}

for biz_key, biz_info in BUSINESS_TYPES.items():
    result = all_type_results[biz_key]
    report['auc_scores'][biz_key] = {
        'label': biz_info['label'],
        'mean': round(float(result['mean_auc']), 4),
        'std':  round(float(result['std_auc']),  4),
    }
    outcome_col = f'outcome_{biz_key}'
    counts = h3_grid[outcome_col].value_counts().to_dict()
    report['outcomes'][biz_key] = {
        'fp': int(counts.get('False Positive (Recommendation)', 0)),
        'tp': int(counts.get('True Positive', 0)),
        'fn': int(counts.get('False Negative', 0)),
        'tn': int(counts.get('True Negative', 0)),
    }
    fp_df = h3_grid[h3_grid[outcome_col] == 'False Positive (Recommendation)'].copy()
    fp_df = fp_df.nlargest(5, f'predicted_prob_{biz_key}')
    recs = []
    for _, row in fp_df.iterrows():
        recs.append({
            'borough':    str(row.get('borough', 'Unknown')),
            'prob':       round(float(row.get(f'predicted_prob_{biz_key}', 0)), 3),
            'population': int(row.get('population', 0)),
            'degree_pct': round(float(row.get('level4_perc', 0)), 1),
            'age_pct':    round(float(row.get('age_16_to_34_perc', 0)), 1),
        })
    report['top_recommendations'][biz_key] = recs

# Top features from already-saved CSV
fi_report = pd.read_csv('data/outputs/feature_importances.csv')
for biz_key in BUSINESS_TYPES:
    top5 = fi_report[fi_report['business_type'] == biz_key].nlargest(5, 'importance')
    report['top_features'][biz_key] = [
        {'feature': str(r['feature']), 'importance': round(float(r['importance']), 4)}
        for _, r in top5.iterrows()
    ]

with open('docs/model_report_data.json', 'w', encoding='utf-8') as _f:
    _json.dump(report, _f, indent=2, ensure_ascii=False)

print('Model report data exported: docs/model_report_data.json')
print(f'  Business types: {len(BUSINESS_TYPES)}')
print(f'  Grid: {report["grid"]["hexagons"]} hexagons, {report["grid"]["boroughs"]} boroughs')
for biz_key in BUSINESS_TYPES:
    label = BUSINESS_TYPES[biz_key]['label']
    auc = report['auc_scores'][biz_key]['mean']
    n_fp = report['outcomes'][biz_key]['fp']
    t = report['thresholds'][biz_key]
    print(f'  {label:25s} AUC={auc:.4f}  FP={n_fp:4d}  threshold={t:.4f}')

# Also write a JS companion so the portfolio works when opened as file://
# (Chrome blocks fetch() between file:// origins -- window._inlineReportData bypasses CORS)
js_content = 'window._inlineReportData = ' + _json.dumps(report, ensure_ascii=False) + ';'
with open('docs/model_report_data.js', 'w', encoding='utf-8') as _jf:
    _jf.write(js_content)
print('Also written: docs/model_report_data.js (file:// safe)')
